# 💰 SalaryLens India — Salary Prediction System v2


## Cell 1 — Install Packages

In [ ]:
import subprocess, sys

PACKAGES = [
    "lightgbm>=4.0", "catboost>=1.2",
    "sentence-transformers>=2.7", "scikit-learn>=1.3",
    "shap>=0.44", "xlrd>=2.0", "pyngrok>=7.0",
]
for pkg in PACKAGES:
    r = subprocess.run([sys.executable,"-m","pip","install",pkg,"-q","--break-system-packages"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        subprocess.run([sys.executable,"-m","pip","install",pkg,"-q"],capture_output=True)
    print(f"  ✅ {pkg}")
print("\n✅ All packages installed")

## Cell 2 — Mount Drive + Config + Load Data



In [ ]:
import os, re, gc, pickle, json, logging, warnings, subprocess, sys
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("salarylens")

# ══════════════════════════════════════════════
# ✏️  CHANGE THIS to your Drive folder name
DRIVE_FOLDER = "salary_prediction_files"
# ══════════════════════════════════════════════

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR   = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
OUT_DIR     = "/content/processed"
MODEL_DIR   = "/content/models"
os.makedirs(OUT_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

FILE_V5     = os.path.join(DRIVE_DIR, "levels_v5_data.csv")
FILE_V4     = os.path.join(DRIVE_DIR, "levels_v4_data.csv")
FILE_V3     = os.path.join(DRIVE_DIR, "levels_v3_data.csv")
FILE_GD     = os.path.join(DRIVE_DIR, "glassdoor_salary_data.csv")
FILE_JP     = os.path.join(DRIVE_DIR, "jobposting_salary_data.csv")
EMB_CACHE   = os.path.join(OUT_DIR,   "embedding_cache.pkl")
UNIFIED_CSV = os.path.join(OUT_DIR,   "unified_salary_data.csv")

# Verify files
print(f"\nChecking: {DRIVE_DIR}\n")
all_ok = True
for label, path in [
    ("levels_v5_data.csv", FILE_V5), ("levels_v4_data.csv", FILE_V4),
    ("levels_v3_data.csv", FILE_V3), ("glassdoor_salary_data.csv", FILE_GD),
    ("jobposting_salary_data.csv", FILE_JP)]:
    ok = os.path.exists(path)
    sz = f"{os.path.getsize(path)/1e6:.1f} MB" if ok else ""
    print(f"  {'✅' if ok else '❌'}  {label:<38} {sz}")
    if not ok: all_ok = False
if not all_ok: raise FileNotFoundError("Fix missing files!")

# ── FIX 1: Rebalanced source weights ─────────────────────────────────────────
# Problem was: 56% job postings dominated → high-end salaries compressed
# Fix: triple Levels.fyi weight (best quality), halve job posting weight
SOURCE_W = {
    "levels_v5": 3.00,   # was 1.00 — tripled (highest quality source)
    "levels_v4": 2.00,   # was 0.75
    "levels_v3": 1.50,   # was 0.60
    "glassdoor": 0.40,   # unchanged — self-reported, noisy
    "jobposting":0.50,   # was 0.70 — reduced to stop dominating
}

SALARY_MIN  = 1_00_000
SALARY_MAX  = 10_00_00_000
USD_INR     = 83.0
YEAR_W      = {2025:1.20,2024:1.10,2023:1.00,2022:0.85,2021:0.70,2020:0.55}
DEF_YW      = 0.50
QUANTILES   = [0.25, 0.50, 0.75]
EMB_DIM     = 32
USE_CB      = True
LGBM_W      = 0.60

# ── FIX 3: Tier-1 MNC set ────────────────────────────────────────────────────
# Problem: Google/Amazon predictions were ₹40L when actual is ₹70–120L
# Fix: explicit binary flag so model learns FAANG pay separately
TIER1_MNCS = {
    "google","amazon","microsoft","meta","apple","netflix","nvidia",
    "qualcomm","adobe","salesforce","uber","linkedin","intuit","atlassian",
    "servicenow","workday","paypal","visa","mastercard",
}
def is_tier1(name):
    if not isinstance(name,str): return 0
    n = name.strip().lower()
    return int(any(t in n for t in TIER1_MNCS))

SENIORITY_MAP = {
    "intern":0,"fresher":0,"trainee":0,
    "l1":1,"e1":1,"sde i":1,"sde1":1,"junior":1,"associate":1,"software engineer 1":1,
    "l2":2,"e2":2,"sde ii":2,"sde2":2,"software engineer 2":2,"mts 1":2,"se2":2,
    "l3":3,"e3":3,"sde iii":3,"sde3":3,"senior software engineer":3,
    "senior engineer":3,"senior":3,"mts 2":3,"se3":3,
    "l4":4,"e4":4,"staff engineer":4,"staff software engineer":4,
    "tech lead":4,"technical lead":4,"lead engineer":4,"mts 3":4,
    "l5":5,"e5":5,"principal engineer":5,"principal sde":5,
    "principal software engineer":5,"distinguished engineer":5,
    "l6":6,"e6":6,"engineering manager":6,"manager":6,"senior manager":6,
    "l7":7,"e7":7,"director":7,"vp":7,"vice president":7,
}
SEN_LABELS = {0:"Intern",1:"Junior",2:"Mid",3:"Senior",
               4:"Staff/Lead",5:"Principal",6:"Manager",7:"Director+"}

CITY_TIER = {
    "bengaluru":1,"bangalore":1,"mumbai":1,"delhi":1,"hyderabad":1,
    "chennai":1,"pune":1,"gurgaon":1,"gurugram":1,"noida":1,"kolkata":1,
    "ahmedabad":2,"jaipur":2,"kochi":2,"chandigarh":2,"coimbatore":2,
    "indore":2,"nagpur":2,"lucknow":2,"visakhapatnam":2,
    "patna":3,"bhubaneswar":3,"dehradun":3,
}
CITY_NORM  = {"bangalore":"bengaluru","gurugram":"gurgaon","bombay":"mumbai",
               "madras":"chennai","calcutta":"kolkata","new delhi":"delhi"}
TECH_HUBS  = {"bengaluru","hyderabad","pune","chennai","noida","gurgaon","mumbai"}
STATE_CITY = {
    "karnataka":"bengaluru","telangana":"hyderabad","maharashtra":"pune",
    "haryana":"gurgaon","delhi":"delhi","uttar pradesh":"noida",
    "tamil nadu":"chennai","west bengal":"kolkata","gujarat":"ahmedabad",
}
CO_TYPE = {
    **{c:"mnc_product" for c in [
        "amazon","google","microsoft","meta","apple","oracle","salesforce","adobe",
        "qualcomm","intel","nvidia","visa","mastercard","cisco","vmware","atlassian",
        "servicenow","nutanix","workday","intuit","expedia","uber","linkedin","sap",
        "ibm","dell technologies","amd","samsung","hpe","micron","paypal",
    ]},
    **{c:"mnc_finance" for c in [
        "goldman sachs","jpmorgan chase","jpmorgan","morgan stanley","blackrock",
        "citi","barclays","hsbc","deutsche bank","ubs","s&p global","bloomberg",
    ]},
    **{c:"consulting" for c in [
        "mckinsey","bcg","bain","deloitte","pwc","kpmg","ey","ernst and young",
        "accenture","thoughtworks","capgemini",
    ]},
    **{c:"it_services" for c in [
        "tata consultancy services","infosys","wipro","hcl","tech mahindra",
        "cognizant","ltimindtree","mphasis","hexaware","zensar",
        "persistent systems","coforge","birlasoft","cyient",
    ]},
    **{c:"indian_product" for c in [
        "flipkart","swiggy","zomato","meesho","razorpay","phonepe","cred",
        "groww","zepto","ola","paytm","freshworks","zoho","sharechat",
        "delhivery","navi","slice","jupiter","media.net",
    ]},
}
TOP_COS     = list(CO_TYPE.keys())
TECH_SKILLS = [
    "python","java","javascript","typescript","react","node","sql","spark",
    "kafka","kubernetes","docker","aws","azure","gcp","machine learning",
    "deep learning","nlp","tensorflow","pytorch","tableau","power bi",
    "golang","scala","django","flask","fastapi","mongodb","postgresql",
    "airflow","snowflake","databricks","llm","generative","transformer",
    "devops","mlops","terraform","android","ios","flutter",
]

LGBM_P = {
    "objective":"quantile","n_estimators":1500,"learning_rate":0.03,
    "max_depth":7,"num_leaves":63,"min_child_samples":20,
    "subsample":0.8,"colsample_bytree":0.8,"reg_alpha":0.1,"reg_lambda":1.0,
    "n_jobs":-1,"random_state":42,"verbose":-1,
}
CB_P = {
    "iterations":1000,"learning_rate":0.05,"depth":7,"l2_leaf_reg":3,
    "random_seed":42,"verbose":200,"early_stopping_rounds":80,
}

# ── Feature functions ─────────────────────────────────────────────────────────
ROLE_RULES = [
    (10,r"engineering manager|software manager",      "Engineering Manager"),
    (20,r"data scientist|ml scientist|research sci",  "Data Scientist"),
    (20,r"data engineer|etl engineer",                "Data Engineer"),
    (20,r"data analyst|bi analyst|analytics eng",     "Data Analyst"),
    (20,r"machine learning engineer|ml engineer|mlops","ML Engineer"),
    (30,r"backend|back.end|server.side",              "Backend Engineer"),
    (30,r"frontend|front.end|ui engineer",            "Frontend Engineer"),
    (30,r"full.?stack|fullstack",                     "Full Stack Engineer"),
    (30,r"android|ios developer|mobile engineer",     "Mobile Engineer"),
    (30,r"devops|site reliability|sre\b|platform eng","DevOps/SRE"),
    (30,r"qa engineer|quality assurance|sdet",        "QA/SDET"),
    (30,r"software development engineer|software engineer|sde\b|swe\b|developer",
        "Software Engineer"),
    (40,r"hardware engineer|vlsi|chip design",        "Hardware Engineer"),
    (50,r"product manager|pm\b|product owner",       "Product Manager"),
    (60,r"management consultant|strategy consultant", "Management Consultant"),
    (80,r"technical program manager|tpm\b",          "Technical Program Manager"),
    (80,r"project manager|scrum master",              "Project Manager"),
    (90,r"solution architect|enterprise architect",   "Solution Architect"),
]
_RC = [(re.compile(p,re.IGNORECASE),r) for _,p,r in sorted(ROLE_RULES,key=lambda x:x[0])]

def norm_role(t, jf=None):
    txt = f"{t or ''} {jf or ''}".lower()
    for pat,role in _RC:
        if pat.search(txt): return role
    return "Other"

def ext_sen(raw, title=""):
    if raw is not None and not (isinstance(raw,float) and np.isnan(raw)):
        s = str(raw).strip().lower()
        if s in SENIORITY_MAP: return SENIORITY_MAP[s]
        if s.isdigit(): return min(7,{1:1,2:2,3:3,4:3,5:4,6:5,7:6,8:6,9:7}.get(int(s),2))
        for kw,v in sorted(SENIORITY_MAP.items(),key=lambda x:-len(x[0])):
            if kw in s: return v
    t = str(title or "").lower()
    if re.search(r"\bintern\b|\bfresher\b",t): return 0
    if re.search(r"\bjunior\b|\bassociate\b",t): return 1
    if re.search(r"\bprincipal\b",t): return 5
    if re.search(r"\bdirector\b|\bvp\b",t): return 7
    if re.search(r"\bstaff\b|\btech lead\b",t): return 4
    if re.search(r"\bsenior\b|\bsr\.?\b",t): return 3
    if re.search(r"\bmanager\b",t): return 6
    return 2

def ext_city(loc):
    if not isinstance(loc,str): return "unknown"
    loc = re.sub(r",?\s*india\s*$","",loc.strip().lower()).strip()
    cand = re.sub(r"\b[a-z]{2}\b$","",loc.split(",")[0].strip()).strip()
    cand = CITY_NORM.get(cand,cand)
    if not cand or cand in("empty","india","n/a",""):
        for st,ct in STATE_CITY.items():
            if st in loc: return ct
        return "unknown"
    return STATE_CITY.get(cand,cand)

def class_co(name):
    if not isinstance(name,str): return "unknown"
    n = name.strip().lower()
    if n in CO_TYPE: return CO_TYPE[n]
    for k,v in CO_TYPE.items():
        if k in n: return v
    if any(kw in n for kw in ["consulting","solutions","technologies","services"]): return "it_services"
    if any(kw in n for kw in ["bank","financial","finance","capital"]): return "mnc_finance"
    return "unknown"

def enc_co(name):
    if not isinstance(name,str): return "other"
    n = name.strip().lower()
    for tc in TOP_COS:
        if tc==n or tc in n: return tc
    return "other"

EDU_MAP = {"phd":4,"doctorate":4,"master":3,"mba":3,"m.tech":3,"pgdm":3,
           "postgraduate":3,"bachelor":2,"undergraduate":2,"b.tech":2,"diploma":1,"high school":0}
def enc_edu(e):
    if not isinstance(e,str): return 2
    el = e.lower()
    for kw,v in sorted(EDU_MAP.items(),key=lambda x:-len(x[0])):
        if kw in el: return v
    return 2

def ext_skills(title):
    text = str(title or "").lower()
    return {"sk_"+s.replace(" ","_").replace(".","").replace("/","_"):int(s in text)
            for s in TECH_SKILLS}

def yw(year):
    if pd.isna(year): return DEF_YW
    return YEAR_W.get(int(year), DEF_YW)

def get_year(df):
    for col in ["OFFER_RECEIVED_DT","SCRAPED_ON_DT","POST_DATE"]:
        if col in df.columns:
            try: return pd.to_datetime(df[col],errors="coerce").dt.year
            except: pass
    return pd.Series([None]*len(df))

# ── Data loaders ─────────────────────────────────────────────────────────────
def load_levels(path, src):
    df = pd.read_csv(path, low_memory=False,
                     usecols=lambda c: c in [
                         "LOCATION","TOTAL_COMP_VAL","EXCHANGE_RATE","COMPANY_NAME",
                         "JOB_TITLE","JOB_LEVEL","JOB_FAMILY","YEARS_OF_EXPERIENCE",
                         "YEARS_AT_COMPANY","YEARS_AT_LEVEL","EDUCATION",
                         "WORK_ARRANGEMENT","OFFER_RECEIVED_DT","SCRAPED_ON_DT"
                     ])
    df = df[df["LOCATION"].str.contains("India",na=False)].copy()
    exr = pd.to_numeric(df.get("EXCHANGE_RATE"),errors="coerce").fillna(USD_INR)
    exr = exr.where(exr>=10, USD_INR)
    yr  = get_year(df)
    out = pd.DataFrame({
        "salary_inr":          pd.to_numeric(df["TOTAL_COMP_VAL"],errors="coerce")*exr,
        "company":             df.get("COMPANY_NAME"),
        "job_title":           df.get("JOB_TITLE"),
        "job_level":           df.get("JOB_LEVEL"),
        "job_family":          df.get("JOB_FAMILY"),
        "location":            df.get("LOCATION"),
        "years_of_experience": pd.to_numeric(df.get("YEARS_OF_EXPERIENCE"),errors="coerce"),
        "years_at_company":    pd.to_numeric(df.get("YEARS_AT_COMPANY"),errors="coerce"),
        "education":           df.get("EDUCATION"),
        "work_arrangement":    df.get("WORK_ARRANGEMENT"),
        "source":              src,
        "sample_weight":       SOURCE_W[src]*yr.apply(yw),
        "data_year":           yr,
    })
    del df; gc.collect()
    return out

def load_gd(path, nrows=30_000):
    df = pd.read_excel(path,engine="xlrd",nrows=nrows) if path.endswith((".xls",".xlsx")) \
         else pd.read_csv(path,low_memory=False,nrows=nrows)
    print(f"  GD shape: {df.shape} | cols: {list(df.columns[:6])}")
    loc_col = next((c for c in ["LOCATION_NAME","LOCATION","location","city"] if c in df.columns),None)
    if loc_col:
        df = df[df[loc_col].str.contains("India",na=False,case=False)].copy()
        print(f"  India rows: {len(df):,}")
    sal_col = next((c for c in ["BASE_PAY_MEDIAN","SALARY_MEDIAN","salary","SALARY"]
                    if c in df.columns),None)
    salary = pd.to_numeric(df[sal_col],errors="coerce") if sal_col else pd.Series([np.nan]*len(df))
    comp_c = next((c for c in ["COMPANY","company","COMPANY_NAME"] if c in df.columns),None)
    titl_c = next((c for c in ["JOB_TITLE_NAME","JOB_TITLE","job_title"] if c in df.columns),None)
    out = pd.DataFrame({
        "salary_inr": salary.values,
        "company":    df[comp_c].values if comp_c else np.nan,
        "job_title":  df[titl_c].values if titl_c else np.nan,
        "job_level":np.nan,"job_family":np.nan,
        "location":   df[loc_col].values if loc_col else "India",
        "years_of_experience":np.nan,"years_at_company":np.nan,
        "education":np.nan,"work_arrangement":np.nan,
        "source":"glassdoor","sample_weight":SOURCE_W["glassdoor"],"data_year":np.nan,
    })
    del df; gc.collect()
    return out

def load_jp(path, nrows=100_000):
    use_cols = ["_SALARY_MIN_RAW","_SALARY_MAX_RAW","_SALARY_RAW",
                "COMPANY_NAME","JOBTITLE_RAW","SENIORITY","CITY","STATE",
                "NORMALIZED_ROLE","HIGHEST_EDUCATION","INDUSTRY","POST_DATE"]
    df = pd.read_csv(path,nrows=nrows,low_memory=False,
                     usecols=lambda c: c in use_cols)
    s_min = pd.to_numeric(df.get("_SALARY_MIN_RAW"),errors="coerce")
    s_max = pd.to_numeric(df.get("_SALARY_MAX_RAW"),errors="coerce")
    s_raw = pd.to_numeric(df.get("_SALARY_RAW"),errors="coerce")
    sal   = np.where(s_min.notna()&s_max.notna(),(s_min+s_max)/2,s_raw)
    city  = df["CITY"].fillna("").astype(str).str.strip().replace("empty","")
    state = df["STATE"].fillna("").astype(str).str.strip()
    loc   = np.where(city!="",city+", "+state+", India",state+", India")
    try:    yr = pd.to_datetime(df.get("POST_DATE"),errors="coerce").dt.year
    except: yr = pd.Series([None]*len(df))
    out = pd.DataFrame({
        "salary_inr":sal,"company":df.get("COMPANY_NAME"),
        "job_title":df.get("JOBTITLE_RAW"),
        "job_level":df.get("SENIORITY").astype(str),
        "job_family":np.nan,"location":loc,
        "years_of_experience":np.nan,"years_at_company":np.nan,
        "education":df.get("HIGHEST_EDUCATION"),"work_arrangement":np.nan,
        "norm_role_pre":df.get("NORMALIZED_ROLE"),
        "sen_pre":pd.to_numeric(df.get("SENIORITY"),errors="coerce"),
        "source":"jobposting","sample_weight":SOURCE_W["jobposting"]*yr.apply(yw),
        "data_year":yr,
    })
    del df; gc.collect()
    return out

def clean(df):
    n0 = len(df)
    df = df.dropna(subset=["salary_inr"])
    df = df[(df["salary_inr"]>=SALARY_MIN)&(df["salary_inr"]<=SALARY_MAX)]
    lo,hi = df["salary_inr"].quantile(0.01),df["salary_inr"].quantile(0.99)
    df = df[(df["salary_inr"]>=lo)&(df["salary_inr"]<=hi)]
    logger.info(f"Cleaned: {n0:,} → {len(df):,}")
    return df.reset_index(drop=True)

# ── Safe reload guard ─────────────────────────────────────────────────────────
# Delete old unified CSV to force reload with new weights
if os.path.exists(UNIFIED_CSV):
    print("Note: Deleting old unified CSV to apply new source weights...")
    os.remove(UNIFIED_CSV)

logger.info("Loading raw files from Drive...")
dfs = []
for src,path in [("levels_v5",FILE_V5),("levels_v4",FILE_V4),("levels_v3",FILE_V3)]:
    d = load_levels(path,src)
    dfs.append(d)
    logger.info(f"  {src}: {len(d):,} rows  weight={SOURCE_W[src]}")
print("\n── Glassdoor (30K sample) ──")
dfs.append(load_gd(FILE_GD, nrows=30_000))
print("\n── Job Postings (100K sample) ──")
dfs.append(load_jp(FILE_JP, nrows=100_000))

unified = pd.concat(dfs,ignore_index=True,sort=False)
del dfs; gc.collect()
unified = clean(unified)
unified.to_csv(UNIFIED_CSV,index=False)
logger.info(f"Saved to {UNIFIED_CSV}")

print(f"\n{'='*50}")
print(f"  Total rows : {len(unified):,}")
print(unified["source"].value_counts().to_string())
s = unified["salary_inr"]/100_000
print(f"\n  Salary: P25=₹{s.quantile(0.25):.1f}L  P50=₹{s.median():.1f}L  P75=₹{s.quantile(0.75):.1f}L")
print(f"{'='*50}")
print("\n✅ Cell 2 complete")

## Cell 3 — Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

logger.info("Building feature matrix with fixes...")
df = unified.copy()

# Role
if "norm_role_pre" in df.columns:
    has = df["norm_role_pre"].notna() & (df["norm_role_pre"]!="")
    df["normalized_role"] = np.where(has,df["norm_role_pre"],
        df.apply(lambda r: norm_role(r.get("job_title"),r.get("job_family")),axis=1))
else:
    df["normalized_role"] = df.apply(
        lambda r: norm_role(r.get("job_title"),r.get("job_family")),axis=1)

# Seniority
if "sen_pre" in df.columns:
    has = df["sen_pre"].notna()
    df["seniority_int"] = np.where(has,
        df["sen_pre"].fillna(2).astype(float).clip(0,7).astype(int),
        df.apply(lambda r: ext_sen(r.get("job_level"),r.get("job_title")),axis=1)).astype(int)
else:
    df["seniority_int"] = df.apply(
        lambda r: ext_sen(r.get("job_level"),r.get("job_title")),axis=1).astype(int)
df["seniority_int"]   = df["seniority_int"].clip(0,7)
df["seniority_label"] = df["seniority_int"].map(SEN_LABELS)

# Location
df["city_norm"]    = df["location"].apply(ext_city)
df["city_tier"]    = df["city_norm"].map(CITY_TIER).fillna(2).astype(int)
df["is_tech_hub"]  = df["city_norm"].isin(TECH_HUBS).astype(int)
df["is_remote"]    = df["location"].fillna("").str.contains("remote",case=False).astype(int)

# Company
df["company_type"] = df["company"].apply(class_co)
df["company_enc"]  = df["company"].apply(enc_co)

# FIX 3: Tier-1 MNC flag
df["is_tier1_mnc"] = df["company"].apply(is_tier1)
print(f"Tier-1 MNC rows: {df['is_tier1_mnc'].sum():,} ({df['is_tier1_mnc'].mean()*100:.1f}%)")

# Education
df["edu_level"] = df["education"].apply(enc_edu)

# Experience
df["yoe"] = pd.to_numeric(df["years_of_experience"],errors="coerce").clip(0,40)
df["yac_raw"] = pd.to_numeric(df.get("years_at_company"),errors="coerce").clip(0,40)

# FIX 2: Replace yac with has_yac binary flag
# Problem: 90%+ of records had yac imputed as 0 → false signal
# Fix: binary flag tells model whether yac is real or imputed
df["has_yac"] = df["yac_raw"].notna().astype(int)
print(f"Records with real yac data: {df['has_yac'].sum():,} ({df['has_yac'].mean()*100:.1f}%)")

yoe_d = {0:0.5,1:1.5,2:3.5,3:6.5,4:9.5,5:13.0,6:12.0,7:16.0}
df["yoe"] = df.apply(
    lambda r: r["yoe"] if pd.notna(r["yoe"]) else yoe_d.get(int(r["seniority_int"]),3.5),axis=1)
df["log_yoe"]   = np.log1p(df["yoe"])
df["exp_bkt"]   = pd.cut(df["yoe"],bins=[-1,2,5,10,15,40],
                          labels=[0,1,2,3,4]).astype(float).fillna(2).astype(int)
df["sen_x_exp"] = df["seniority_int"] * df["log_yoe"]

# Skills
sk_df      = df["job_title"].apply(ext_skills).apply(pd.Series)
skill_cols = [c for c in sk_df.columns if sk_df[c].sum()>=100]
df         = pd.concat([df,sk_df[skill_cols]],axis=1)
del sk_df; gc.collect()

# Temporal
df["yr_feat"] = df["data_year"].apply(lambda y: YEAR_W.get(int(y),DEF_YW) if pd.notna(y) else DEF_YW)

df = df.dropna(subset=["salary_inr"]).reset_index(drop=True)

print(f"\n  Rows: {len(df):,}   Skill features: {len(skill_cols)}")
print("\n  Role (top 6):")
print(df["normalized_role"].value_counts().head(6).to_string())
print("\n  Seniority:")
print(df["seniority_label"].value_counts().sort_index().to_string())
print("\n  Company type:")
print(df["company_type"].value_counts().to_string())
print("\n  Fixes applied:")
print(f"    ✅ Fix 1: Source weights — Levels.fyi 3x, JobPosting 0.5x")
print(f"    ✅ Fix 2: has_yac binary flag (replaces noisy yac column)")
print(f"    ✅ Fix 3: is_tier1_mnc flag ({df['is_tier1_mnc'].sum():,} FAANG rows flagged)")
print("\n✅ Cell 3 complete")

## Cell 4 — Embeddings + Train Models

In [ ]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool

# FIX 2+3: Updated feature columns
# Removed: yac, yal, log_yac (noisy imputed columns)
# Added:   has_yac (binary), is_tier1_mnc (FAANG signal)
CAT_COLS  = ["normalized_role","company_type","company_enc","city_tier","edu_level"]
NUM_COLS  = [
    "seniority_int","yoe","has_yac","is_tech_hub","is_remote",
    "is_tier1_mnc",   # NEW: separates FAANG from other MNCs
    "log_yoe","exp_bkt","sen_x_exp","yr_feat",
    # removed: yac, yal, log_yac (were imputed zeros, adding noise)
]
BASE_COLS = CAT_COLS + [c for c in NUM_COLS if c not in CAT_COLS] + skill_cols

label_enc = {}
for col in CAT_COLS:
    if col not in df.columns: df[col]="unknown"
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].fillna("unknown").astype(str))
    label_enc[col] = le

for col in NUM_COLS+skill_cols:
    if col in df.columns and df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

target  = np.log1p(df["salary_inr"].values)
wts     = df["sample_weight"].fillna(1.0).values
strat   = df["seniority_int"].fillna(2).astype(int).values
idx_tr, idx_va = train_test_split(np.arange(len(df)),test_size=0.20,
                                    random_state=42,stratify=strat)
y_tr, y_va = target[idx_tr], target[idx_va]
w_tr       = wts[idx_tr]
logger.info(f"Train: {len(idx_tr):,}  Val: {len(idx_va):,}  Features: {len(BASE_COLS)+EMB_DIM}")

# Embeddings
def emb_text(row):
    parts = [str(row.get(c,"") or "") for c in
             ["normalized_role","job_title","seniority_label"]
             if pd.notna(row.get(c)) and str(row.get(c,"")).strip()]
    return " ".join(parts) or "Software Engineer"

texts = df.apply(emb_text,axis=1).tolist()
cache = {}
if os.path.exists(EMB_CACHE):
    with open(EMB_CACHE,"rb") as f: cache = pickle.load(f)
    logger.info(f"Cache: {len(cache)} entries")

to_compute = [t for t in list(dict.fromkeys(texts)) if t not in cache]
if to_compute:
    logger.info(f"Computing {len(to_compute):,} embeddings...")
    em = SentenceTransformer("all-MiniLM-L6-v2")
    vecs = em.encode(to_compute,batch_size=256,show_progress_bar=True,convert_to_numpy=True)
    cache.update({t:v for t,v in zip(to_compute,vecs)})
    with open(EMB_CACHE,"wb") as f: pickle.dump(cache,f)
    del em,vecs; gc.collect()
else:
    logger.info("All embeddings from cache ✅")

raw_emb = np.array([cache[t] for t in texts],dtype=np.float32)
pca = PCA(n_components=EMB_DIM,random_state=42)
pca.fit(raw_emb[idx_tr])
logger.info(f"PCA: 384D→{EMB_DIM}D, {pca.explained_variance_ratio_.sum()*100:.1f}% variance")

emb_cols   = [f"e{i}" for i in range(EMB_DIM)]
tr_emb_df  = pd.DataFrame(pca.transform(raw_emb[idx_tr]),columns=emb_cols)
va_emb_df  = pd.DataFrame(pca.transform(raw_emb[idx_va]),columns=emb_cols)
del raw_emb; gc.collect()

X_base = df[BASE_COLS].copy()
X_tr   = pd.concat([X_base.iloc[idx_tr].reset_index(drop=True),tr_emb_df],axis=1)
X_va   = pd.concat([X_base.iloc[idx_va].reset_index(drop=True),va_emb_df],axis=1)
FEATS  = BASE_COLS + emb_cols

# LightGBM
lgbm_m = {}
for q in QUANTILES:
    logger.info(f"LGBM P{int(q*100)}...")
    m = lgb.LGBMRegressor(**{**LGBM_P,"alpha":q})
    m.fit(X_tr[FEATS],y_tr,sample_weight=w_tr,
          eval_set=[(X_va[FEATS],y_va)],
          callbacks=[lgb.early_stopping(100,verbose=False),lgb.log_evaluation(300)])
    lgbm_m[q] = m
    logger.info(f"  P{int(q*100)} done — iter:{m.best_iteration_}")
print("✅ LightGBM done")

# CatBoost
cb_m = {}
if USE_CB:
    df_cb = df.copy()
    for col in CAT_COLS:
        df_cb[col] = label_enc[col].inverse_transform(df_cb[col].astype(int))
    Xcb = df_cb[BASE_COLS].copy(); del df_cb; gc.collect()
    Xcb_tr = pd.concat([Xcb.iloc[idx_tr].reset_index(drop=True),tr_emb_df],axis=1)
    Xcb_va = pd.concat([Xcb.iloc[idx_va].reset_index(drop=True),va_emb_df],axis=1)
    cat_idx= [Xcb_tr.columns.tolist().index(c) for c in CAT_COLS if c in Xcb_tr.columns]
    del Xcb; gc.collect()
    for q in QUANTILES:
        logger.info(f"CatBoost P{int(q*100)}...")
        p = {**CB_P,"loss_function":f"Quantile:alpha={q}","eval_metric":f"Quantile:alpha={q}"}
        m = CatBoostRegressor(**p)
        m.fit(Pool(Xcb_tr,y_tr,weight=w_tr,cat_features=cat_idx),
              eval_set=Pool(Xcb_va,y_va,cat_features=cat_idx),use_best_model=True)
        cb_m[q] = m
    print("✅ CatBoost done")

def blend(q,Xlg,Xcb=None):
    lp = lgbm_m[q].predict(Xlg[FEATS])
    if cb_m and q in cb_m and Xcb is not None:
        return LGBM_W*lp+(1-LGBM_W)*cb_m[q].predict(Xcb)
    return lp

# FIX 4: Sort-based monotonicity — mathematically guaranteed 0 crossings
def mono(p25,p50,p75):
    stack = np.sort(np.stack([np.array(p25),np.array(p50),np.array(p75)],axis=1),axis=1)
    return stack[:,0], stack[:,1], stack[:,2]

Xcbv = Xcb_va if USE_CB and cb_m else None
lp25,lp50,lp75 = blend(0.25,X_va,Xcbv),blend(0.50,X_va,Xcbv),blend(0.75,X_va,Xcbv)
lp25,lp50,lp75 = mono(lp25,lp50,lp75)
p25v,p50v,p75v = np.expm1(lp25),np.expm1(lp50),np.expm1(lp75)
y_va_act       = np.expm1(y_va)
crossings = int(((p25v>p50v)|(p50v>p75v)).sum())
print(f"Crossings: {crossings}  (target: 0)")
print("\n✅ Cell 4 complete")

## Cell 5 — Evaluate + Save

In [ ]:
eps = 1e-6
metrics = {
    "mae_lpa":    float(np.mean(np.abs(y_va_act-p50v))/100_000),
    "mape_pct":   float(np.mean(np.abs((y_va_act-p50v)/(y_va_act+eps)))*100),
    "calibration":float((y_va_act<=p50v).mean()*100),
    "range_acc":  float(((y_va_act>=p25v)&(y_va_act<=p75v)).mean()*100),
    "range_width":float(((p75v-p25v)/(p50v+eps)).mean()*100),
    "crossings":  int(((p25v>p50v)|(p50v>p75v)).sum()),
}

print("\n" + "═"*52)
print("  EVALUATION — SalaryLens India v2")
print("  (after all 4 fixes)")
print("═"*52)
print(f"  MAE            ₹{metrics['mae_lpa']:.2f} LPA")
print(f"  MAPE           {metrics['mape_pct']:.1f}%")
print(f"  Calibration    {metrics['calibration']:.1f}%   target=50%")
print(f"  Range Accuracy {metrics['range_acc']:.1f}%   target=45-60%")
print(f"  Range Width    {metrics['range_width']:.1f}% of median")
print(f"  Crossings      {metrics['crossings']}          target=0")
print("═"*52)

vdf = df.iloc[idx_va].copy().reset_index(drop=True)
vdf["yt"],vdf["p25"],vdf["p50"],vdf["p75"]=y_va_act,p25v,p50v,p75v
print("\n  By Seniority:")
for seg,g in vdf.groupby("seniority_label"):
    acc=((g.yt>=g.p25)&(g.yt<=g.p75)).mean()*100
    mae=abs(g.yt-g.p50).mean()/100_000
    print(f"    {seg:<16} {acc:5.1f}%  MAE=₹{mae:.1f}L  n={len(g):,}")

print("\n  By Company Type:")
for seg,g in vdf.groupby("company_type"):
    le = label_enc.get("company_type")
    label = le.inverse_transform([seg])[0] if le else str(seg)
    acc=((g.yt>=g.p25)&(g.yt<=g.p75)).mean()*100
    print(f"    {label:<22} {acc:5.1f}%  n={len(g):,}")

# SHAP
shap_sum={}
try:
    import shap
    sidx=np.random.choice(len(X_va),min(300,len(X_va)),replace=False)
    sv  =shap.TreeExplainer(lgbm_m[0.50]).shap_values(X_va[FEATS].iloc[sidx])
    fi  =pd.Series(np.abs(sv).mean(axis=0),index=FEATS).sort_values(ascending=False)
    shap_sum=fi.to_dict()
    print("\n  Top 10 SHAP features:")
    for f,v in fi.head(10).items():
        print(f"    {f:<30} {'█'*int(v/fi.max()*20)} {v:.4f}")
except Exception as e:
    print(f"  SHAP skipped: {e}")

# Save
art_path = os.path.join(MODEL_DIR,"salary_model_artifacts.pkl")
artifacts = {
    "lgbm_models":lgbm_m,"catboost_models":cb_m if USE_CB else {},
    "label_enc":label_enc,"pca":pca,"feats":FEATS,"cat_cols":CAT_COLS,
    "skill_cols":skill_cols,"emb_cols":emb_cols,"shap":shap_sum,"metrics":metrics,
    "emb_cache":EMB_CACHE,"sen_labels":SEN_LABELS,"city_tier":CITY_TIER,
    "tech_hubs":list(TECH_HUBS),"co_type":CO_TYPE,"top_cos":TOP_COS,
    "tech_skills":TECH_SKILLS,"tier1_mncs":list(TIER1_MNCS),
}
with open(art_path,"wb") as f: pickle.dump(artifacts,f)

drive_bk = os.path.join(DRIVE_DIR,"saved_models")
os.makedirs(drive_bk,exist_ok=True)
import shutil
shutil.copy(art_path,os.path.join(drive_bk,"salary_model_artifacts_v2.pkl"))

meta={"trained_at":datetime.now().isoformat(),"n_train":int(len(idx_tr)),
      "n_val":int(len(idx_va)),"n_features":len(FEATS),
      "fixes_applied":["source_reweight_3x","has_yac_binary","is_tier1_mnc","sort_mono"],
      "evaluation":{k:round(float(v),4) for k,v in metrics.items()}}
with open(os.path.join(MODEL_DIR,"metadata.json"),"w") as f: json.dump(meta,f,indent=2)

print(f"\n✅ Artifacts saved: {art_path} ({os.path.getsize(art_path)/1e6:.1f} MB)")
print(f"✅ Drive backup:    {drive_bk}/salary_model_artifacts_v2.pkl")

## Cell 6 — Inference Test

In [ ]:
def predict(job_title, company, yoe_val, city, sen_override=None):
    role  = norm_role(job_title)
    si    = int(np.clip(sen_override if sen_override is not None
                        else ext_sen(None,job_title),0,7))
    ct    = ext_city(city+", India")
    yoe_f = float(yoe_val)
    logy  = np.log1p(yoe_f)
    eb    = int(pd.cut([yoe_f],bins=[-1,2,5,10,15,40],labels=[0,1,2,3,4])[0] or 2)
    skl   = ext_skills(job_title)

    row = {
        "normalized_role":role, "company_type":class_co(company),
        "company_enc":enc_co(company), "city_tier":CITY_TIER.get(ct,2),
        "edu_level":2, "seniority_int":si, "yoe":yoe_f,
        "has_yac":0,           # no yac available at inference
        "is_tech_hub":int(ct in TECH_HUBS), "is_remote":0,
        "is_tier1_mnc":is_tier1(company),   # FAANG flag
        "log_yoe":logy, "exp_bkt":eb, "sen_x_exp":si*logy,
        "yr_feat":YEAR_W.get(2025,1.2), **skl,
    }

    rl = pd.DataFrame([row])
    for col in CAT_COLS:
        le  = label_enc[col]
        val = str(rl.get(col,pd.Series(["unknown"])).iloc[0])
        val = val if val in set(le.classes_) else le.classes_[0]
        rl[col] = le.transform([val])[0]
    for col in FEATS:
        if col not in rl.columns: rl[col] = 0.0

    et  = f"{role} {job_title} {SEN_LABELS.get(si,'Mid')}"
    rev = cache.get(et)
    if rev is None:
        rev = SentenceTransformer("all-MiniLM-L6-v2").encode([et])[0]
    ec  = pca.transform(rev.reshape(1,-1))
    for i,col in enumerate(emb_cols): rl[col] = ec[0,i]

    l25=lgbm_m[0.25].predict(rl[FEATS])[0]
    l50=lgbm_m[0.50].predict(rl[FEATS])[0]
    l75=lgbm_m[0.75].predict(rl[FEATS])[0]
    if cb_m:
        rc=pd.DataFrame([row])
        for col in FEATS:
            if col not in rc.columns: rc[col]=0.0
        for i,col in enumerate(emb_cols): rc[col]=ec[0,i]
        l25=LGBM_W*l25+(1-LGBM_W)*cb_m[0.25].predict(rc)[0]
        l50=LGBM_W*l50+(1-LGBM_W)*cb_m[0.50].predict(rc)[0]
        l75=LGBM_W*l75+(1-LGBM_W)*cb_m[0.75].predict(rc)[0]

    a,b,c = mono([l25],[l50],[l75])
    conf  = "high" if class_co(company)!="unknown" and role!="Other" else "medium"
    return {"p25":round(np.expm1(a[0])/1e5,1),"p50":round(np.expm1(b[0])/1e5,1),
            "p75":round(np.expm1(c[0])/1e5,1),"role":role,
            "seniority":SEN_LABELS.get(si,"Mid"),"city":ct,"conf":conf,
            "is_tier1":is_tier1(company)}

# Sanity check vs known market benchmarks
GROUND_TRUTH = [
    ("SDE II",                  "Amazon",    3,  "Bengaluru",  (40,60)),
    ("Senior Data Scientist",   "Flipkart",  5,  "Bengaluru",  (30,55)),
    ("Data Analyst",            "Deloitte",  2,  "Gurgaon",    (8,18)),
    ("Software Engineer",       "Infosys",   1,  "Pune",       (4,10)),
    ("Principal ML Engineer",   "Google",    10, "Bengaluru",  (70,120)),
    ("Product Manager",         "Meesho",    4,  "Bengaluru",  (25,50)),
    ("Junior Software Engineer","TCS",       1,  "Bengaluru",  (3,8)),
    ("DevOps Engineer",         "Microsoft", 6,  "Hyderabad",  (25,50)),
    ("Data Engineer",           "Razorpay",  3,  "Bengaluru",  (15,35)),
]

print(f"\n{'═'*72}")
print(f"  SANITY CHECK — Ground Truth vs Predictions")
print(f"  Source: Levels.fyi India / industry benchmarks")
print(f"{'─'*72}")
print(f"  {'Role':<28}{'Company':<12}{'P50 Pred':>9}  {'Expected':>14}  Status")
print(f"{'─'*72}")
passed=0
for role,co,yoe,city,(lo,hi) in GROUND_TRUTH:
    r   = predict(role,co,yoe,city)
    p50 = r["p50"]
    ok  = lo<=p50<=hi
    if ok: passed+=1
    t1  = " [T1]" if r["is_tier1"] else ""
    print(f"  {role:<28}{co+t1:<17}{p50:>7.1f}L  {f'₹{lo}–{hi}L':>14}  {'✅' if ok else '⚠️'}")
print(f"{'═'*72}")
print(f"  Pass rate: {passed}/{len(GROUND_TRUTH)} ({passed/len(GROUND_TRUTH)*100:.0f}%)")

# Full inference table
TESTS=[
    ("Senior Data Scientist","Flipkart",5,"Bengaluru"),
    ("SDE II","Amazon",3,"Hyderabad"),
    ("Data Analyst","Deloitte",2,"Gurgaon"),
    ("Principal ML Engineer","Google",10,"Bengaluru"),
    ("Junior Software Engineer","Infosys",1,"Pune"),
    ("Product Manager","Meesho",4,"Bengaluru"),
    ("DevOps Engineer","Microsoft",6,"Hyderabad"),
    ("Data Engineer","Razorpay",3,"Bengaluru"),
]
print(f"\n{'─'*90}")
print(f"  {'Job Title':<28}{'Company':<12}{'YoE':>3}  {'P25':>7}  {'P50':>7}  {'P75':>7}  Conf")
print(f"{'─'*90}")
for jt,co,yoe_v,city in TESTS:
    r=predict(jt,co,yoe_v,city)
    print(f"  {jt:<28}{co:<12}{yoe_v:>3}  ₹{r['p25']:>5.1f}L  ₹{r['p50']:>5.1f}L  ₹{r['p75']:>5.1f}L  [{r['conf']}]")
print(f"{'─'*90}")
print(f"\n🏁 PIPELINE COMPLETE — v2 with all fixes")
print(f"   MAE: ₹{metrics['mae_lpa']:.2f} LPA  |  Calibration: {metrics['calibration']:.1f}%  |  Range Acc: {metrics['range_acc']:.1f}%  |  Crossings: {metrics['crossings']}")

## Cell 7 — Launch Streamlit App



In [ ]:
import os, subprocess, sys, time

# ══════════════════════════════════════════════════════
NGROK_TOKEN = "PASTE_YOUR_TOKEN_HERE"  # get free token at dashboard.ngrok.com
# ══════════════════════════════════════════════════════

APP_PATH = "/content/models/streamlit_app.py"

app_lines = [
    'import os, pickle, warnings, re\n',
    'import numpy as np\n',
    'import pandas as pd\n',
    'import streamlit as st\n',
    'from sentence_transformers import SentenceTransformer\n',
    'warnings.filterwarnings("ignore")\n',
    '\n',
    'st.set_page_config(\n',
    '    page_title="SalaryLens India",\n',
    '    page_icon="💰",\n',
    '    layout="wide",\n',
    '    initial_sidebar_state="collapsed"\n',
    ')\n',
    '\n',
    'st.markdown("""\n',
    '<style>\n',
    "@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');\n",
    'html,body,[class*="css"]{font-family:\'Plus Jakarta Sans\',sans-serif}\n',
    'section[data-testid="stSidebar"]{display:none}\n',
    '#MainMenu,footer,header{visibility:hidden}\n',
    '.stProgress>div>div{background:linear-gradient(90deg,#6366f1,#8b5cf6)}\n',
    'div[data-testid="stHorizontalBlock"]{gap:0.8rem}\n',
    '</style>\n',
    '""", unsafe_allow_html=True)\n',
    '\n',
    '@st.cache_resource\n',
    'def load_arts():\n',
    '    with open("/content/models/salary_model_artifacts.pkl","rb") as f:\n',
    '        return pickle.load(f)\n',
    '\n',
    'arts        = load_arts()\n',
    'lgbm_m      = arts["lgbm_models"]\n',
    'cb_m        = arts.get("catboost_models",{})\n',
    'label_enc   = arts["label_enc"]\n',
    'pca         = arts["pca"]\n',
    'FEATS       = arts["feats"]\n',
    'CAT_COLS    = arts["cat_cols"]\n',
    'skill_cols  = arts["skill_cols"]\n',
    'emb_cols    = arts["emb_cols"]\n',
    'shap_imp    = arts.get("shap",{})\n',
    'SEN_LABELS  = arts["sen_labels"]\n',
    'CITY_TIER   = arts["city_tier"]\n',
    'TECH_HUBS   = set(arts["tech_hubs"])\n',
    'CO_TYPE     = arts["co_type"]\n',
    'TOP_COS     = arts["top_cos"]\n',
    'TECH_SKILLS = arts["tech_skills"]\n',
    'YEAR_W      = {2025:1.20,2024:1.10,2023:1.00,2022:0.85,2021:0.70,2020:0.55}\n',
    '\n',
    '# ── Extended company classification ─────────────────────────────────────────\n',
    '# CO_TYPE loaded from artifacts covers 87 named companies.\n',
    '# EXTENDED_CO_TYPE covers additional companies that appear in training data\n',
    '# but were not in CO_TYPE during training — adding them here improves\n',
    '# inference routing to the correct company type bucket (no retraining needed).\n',
    'EXTENDED_CO_TYPE = {\n',
    '    # IT Services aliases\n',
    '    "tcs":"it_services","t c s":"it_services",\n',
    '    "l&t technology services":"it_services","ltts":"it_services",\n',
    '    "niit technologies":"it_services","mastech":"it_services",\n',
    '    # Indian Product missing\n',
    '    "nykaa":"indian_product","browserstack":"indian_product",\n',
    '    "zerodha":"indian_product","upstox":"indian_product",\n',
    '    "oyo":"indian_product","cars24":"indian_product",\n',
    '    "shiprocket":"indian_product","lenskart":"indian_product",\n',
    '    "makemytrip":"indian_product","mmt":"indian_product",\n',
    '    "ixigo":"indian_product","rapido":"indian_product",\n',
    '    "smallcase":"indian_product","juspay":"indian_product",\n',
    '    "setu":"indian_product","decentro":"indian_product",\n',
    '    "khatabook":"indian_product","dunzo":"indian_product",\n',
    '    "spinny":"indian_product","udaan":"indian_product",\n',
    '    "unacademy":"indian_product","byjus":"indian_product",\n',
    '    "byju\'s":"indian_product","fampay":"indian_product",\n',
    '    "niyo":"indian_product","fi money":"indian_product",\n',
    '    "stashfin":"indian_product","rupeek":"indian_product",\n',
    '    "postman":"indian_product","hasura":"indian_product",\n',
    '    # MNC Product missing\n',
    '    "netflix":"mnc_product","airbnb":"mnc_product",\n',
    '    "stripe":"mnc_product","datadog":"mnc_product",\n',
    '    "cloudflare":"mnc_product","gitlab":"mnc_product",\n',
    '    "github":"mnc_product","confluent":"mnc_product",\n',
    '    "elastic":"mnc_product","okta":"mnc_product",\n',
    '    "pagerduty":"mnc_product","zendesk":"mnc_product",\n',
    '    "twilio":"mnc_product","grafana":"mnc_product",\n',
    '    "hashicorp":"mnc_product","mongodb":"mnc_product",\n',
    '    "snowflake":"mnc_product","databricks":"mnc_product",\n',
    '    "figma":"mnc_product","notion":"mnc_product",\n',
    '    "slack":"mnc_product","zoom":"mnc_product",\n',
    '    "docusign":"mnc_product","hubspot":"mnc_product",\n',
    '    "jfrog":"mnc_product","fastly":"mnc_product",\n',
    '    # Finance missing\n',
    '    "hdfc":"mnc_finance","icici":"mnc_finance",\n',
    '    "kotak":"mnc_finance","axis bank":"mnc_finance",\n',
    '    "standard chartered":"mnc_finance",\n',
    '    "jp morgan":"mnc_finance","wells fargo":"mnc_finance",\n',
    '    "bank of america":"mnc_finance","boa":"mnc_finance",\n',
    '    "idfc":"mnc_finance","rbl bank":"mnc_finance",\n',
    '    "yes bank":"mnc_finance","sbi":"mnc_finance",\n',
    '    "paytm payments bank":"mnc_finance",\n',
    '    # Consulting missing\n',
    '    "booz allen":"consulting","oliver wyman":"consulting",\n',
    '    "pa consulting":"consulting","ey-parthenon":"consulting",\n',
    '    "accenture song":"consulting","zs associates":"consulting",\n',
    '    "mu sigma":"consulting","fractal analytics":"consulting",\n',
    '}\n',
    '\n',
    '# Merge: EXTENDED takes priority over CO_TYPE for overlapping keys\n',
    '_FULL_CO_TYPE = {**CO_TYPE, **EXTENDED_CO_TYPE}\n',
    '\n',
    'TIER1_MNCS = {\n',
    '    "google","amazon","microsoft","meta","apple","netflix","nvidia",\n',
    '    "qualcomm","adobe","salesforce","uber","linkedin","intuit","atlassian",\n',
    '    "servicenow","workday","paypal","visa","mastercard",\n',
    '    # Added missing Tier-1s\n',
    '    "airbnb","stripe","datadog","cloudflare","snowflake","databricks",\n',
    '}\n',
    '\n',
    'def is_tier1(name):\n',
    '    if not isinstance(name,str): return 0\n',
    '    return int(any(t in name.strip().lower() for t in TIER1_MNCS))\n',
    '\n',
    '# Smart keyword-based fallback for truly unknown companies\n',
    '# Catches patterns the explicit list misses\n',
    '_IT_KEYWORDS    = ["technologies","technology","tech","solutions","systems",\n',
    '                    "infotech","software services","consulting services","outsourcing"]\n',
    '_FINANCE_KW     = ["bank","finance","financial","capital","investment","securities",\n',
    '                    "insurance","asset management","wealth","trading","fintech"]\n',
    '_PRODUCT_KW     = ["labs","ai","studio","platform","cloud","data","analytics",\n',
    '                    "intelligence","automation","robotics","networks"]\n',
    '_STARTUP_SIGNALS= ["startup","ventures","inc","corp","co.","pvt","limited","ltd"]\n',
    '\n',
    'def _keyword_classify(n):\n',
    '    """Classify by name keywords when explicit lookup fails."""\n',
    '    if any(kw in n for kw in _FINANCE_KW):   return "mnc_finance"\n',
    '    if any(kw in n for kw in _IT_KEYWORDS):   return "it_services"\n',
    '    if any(kw in n for kw in _PRODUCT_KW):    return "indian_product"\n',
    '    return "unknown"\n',
    '\n',
    'CITY_NORM  = {"bangalore":"bengaluru","gurugram":"gurgaon","bombay":"mumbai",\n',
    '               "madras":"chennai","calcutta":"kolkata","new delhi":"delhi"}\n',
    'STATE_CITY = {"karnataka":"bengaluru","telangana":"hyderabad","maharashtra":"pune",\n',
    '               "haryana":"gurgaon","delhi":"delhi","uttar pradesh":"noida",\n',
    '               "tamil nadu":"chennai","west bengal":"kolkata","gujarat":"ahmedabad"}\n',
    '\n',
    'ROLE_RULES = [\n',
    '    (10,r"engineering manager|software manager","Engineering Manager"),\n',
    '    (20,r"data scientist|ml scientist|research sci","Data Scientist"),\n',
    '    (20,r"data engineer|etl engineer","Data Engineer"),\n',
    '    (20,r"data analyst|bi analyst|analytics eng","Data Analyst"),\n',
    '    (20,r"machine learning engineer|ml engineer|mlops","ML Engineer"),\n',
    '    (30,r"backend|back.end","Backend Engineer"),\n',
    '    (30,r"frontend|front.end","Frontend Engineer"),\n',
    '    (30,r"full.?stack|fullstack","Full Stack Engineer"),\n',
    '    (30,r"devops|site reliability","DevOps/SRE"),\n',
    '    (30,r"software development engineer|software engineer|sde|swe|developer","Software Engineer"),\n',
    '    (40,r"hardware engineer|vlsi","Hardware Engineer"),\n',
    '    (50,r"product manager|product owner","Product Manager"),\n',
    '    (80,r"project manager|scrum master","Project Manager"),\n',
    '    (90,r"solution architect|enterprise architect","Solution Architect"),\n',
    '    (100,r"recruiter|talent acquisition|hr manager|human resource|people ops|hrbp","HR/Recruiter"),\n',
    ']\n',
    '_RC = [(re.compile(p,re.IGNORECASE),r) for _,p,r in sorted(ROLE_RULES,key=lambda x:x[0])]\n',
    '\n',
    'def norm_role(t):\n',
    '    for pat,role in _RC:\n',
    '        if pat.search(str(t or "").lower()): return role\n',
    '    return "Other"\n',
    '\n',
    'def ext_sen(title=""):\n',
    '    t=str(title or "").lower()\n',
    '    if re.search(r"\\bintern\\b|\\bfresher\\b",t): return 0\n',
    '    if re.search(r"\\bjunior\\b|\\bassociate\\b",t): return 1\n',
    '    if re.search(r"\\bprincipal\\b",t): return 5\n',
    '    if re.search(r"\\bdirector\\b|\\bvp\\b",t): return 7\n',
    '    if re.search(r"\\bstaff\\b|\\btech lead\\b",t): return 4\n',
    '    if re.search(r"\\bsenior\\b|\\bsr\\.?\\b",t): return 3\n',
    '    if re.search(r"\\bmanager\\b",t): return 6\n',
    '    return 2\n',
    '\n',
    'def ext_city(loc):\n',
    '    if not isinstance(loc,str): return "unknown"\n',
    '    loc=re.sub(r",?\\s*india\\s*$","",loc.strip().lower()).strip()\n',
    '    cand=re.sub(r"\\b[a-z]{2}\\b$","",loc.split(",")[0].strip()).strip()\n',
    '    cand=CITY_NORM.get(cand,cand)\n',
    '    if not cand or cand in("empty","india","n/a",""):\n',
    '        for s,c in STATE_CITY.items():\n',
    '            if s in loc: return c\n',
    '        return "unknown"\n',
    '    return STATE_CITY.get(cand,cand)\n',
    '\n',
    'def class_co(name):\n',
    '    """3-layer lookup: exact match → substring → keyword heuristic."""\n',
    '    if not isinstance(name,str): return "unknown"\n',
    '    n = name.strip().lower()\n',
    '    if n in _FULL_CO_TYPE: return _FULL_CO_TYPE[n]\n',
    '    for k,v in _FULL_CO_TYPE.items():\n',
    '        if k in n: return v\n',
    '    return _keyword_classify(n)\n',
    '\n',
    'def enc_co(name):\n',
    '    """Return specific company encoding if known, else other."""\n',
    '    if not isinstance(name,str): return "other"\n',
    '    n = name.strip().lower()\n',
    '    all_cos = list(CO_TYPE.keys()) + list(EXTENDED_CO_TYPE.keys())\n',
    '    for tc in all_cos:\n',
    '        if tc == n or tc in n: return tc\n',
    '    return "other"\n',
    '\n',
    'def ext_skills(title):\n',
    '    text=str(title or "").lower()\n',
    '    return {"sk_"+s.replace(" ","_").replace(".","").replace("/","_"):int(s in text)\n',
    '            for s in TECH_SKILLS}\n',
    '\n',
    'def mono(p25,p50,p75):\n',
    '    stack=np.sort(np.stack([np.array(p25),np.array(p50),np.array(p75)],axis=1),axis=1)\n',
    '    return stack[:,0],stack[:,1],stack[:,2]\n',
    '\n',
    '@st.cache_resource\n',
    'def get_emb(): return SentenceTransformer("all-MiniLM-L6-v2")\n',
    '\n',
    '\n',
    '# ── SIMILAR COMPANY SUGGESTIONS ───────────────────────────────────────────────\n',
    '# When a company is fully unknown, suggest the closest known companies\n',
    '# based on name similarity + inferred type signals from the query text\n',
    '\n',
    '_ALL_KNOWN_COS = sorted(set(list(CO_TYPE.keys()) + list(EXTENDED_CO_TYPE.keys())))\n',
    '\n',
    'def _suggest_similar(company_name: str, n: int = 5) -> list:\n',
    '    """\n',
    '    Returns list of (company_name, company_type) tuples most similar\n',
    '    to the unknown company. Uses two signals:\n',
    '    1. String token overlap (catches partial name matches)\n',
    '    2. Inferred category from keyword (suggests same-type companies)\n',
    '    """\n',
    '    if not isinstance(company_name, str): return []\n',
    '    q = company_name.strip().lower()\n',
    '\n',
    '    # Signal 1: token overlap score\n',
    '    q_tokens = set(q.replace("-"," ").replace("."," ").split())\n',
    '    scored = []\n',
    '    for co in _ALL_KNOWN_COS:\n',
    '        co_tokens = set(co.replace("-"," ").replace("."," ").split())\n',
    '        overlap   = len(q_tokens & co_tokens)\n',
    '        # Bonus for substring match\n',
    '        substr    = 2 if (q in co or co in q) else 0\n',
    '        scored.append((overlap + substr, co))\n',
    '\n',
    '    # Signal 2: infer category and boost same-type companies\n',
    '    inferred_type = _keyword_classify(q)\n',
    '    boosted = []\n',
    '    for score, co in scored:\n',
    '        co_type = _FULL_CO_TYPE.get(co, "unknown")\n',
    '        type_bonus = 1 if co_type == inferred_type else 0\n',
    '        boosted.append((score + type_bonus, co, co_type))\n',
    '\n',
    '    # Sort by score desc, then alphabetically\n',
    '    boosted.sort(key=lambda x: (-x[0], x[1]))\n',
    '\n',
    '    # Return top n with score > 0, otherwise return top n by category\n',
    '    results = [(co, ct) for _, co, ct in boosted if _ > 0][:n]\n',
    '    if len(results) < n:\n',
    '        # Fill with same-category companies if not enough token matches\n',
    '        same_type = [\n',
    '            (co, ct) for _, co, ct in boosted\n',
    '            if ct == inferred_type and (co, ct) not in results\n',
    '        ]\n',
    '        results = (results + same_type)[:n]\n',
    '    if len(results) < n:\n',
    '        # Final fallback — just top companies\n',
    '        top_fallback = [\n',
    '            (co, ct) for _, co, ct in boosted\n',
    '            if (co, ct) not in results\n',
    '        ][:n-len(results)]\n',
    '        results = results + top_fallback\n',
    '\n',
    '    return results[:n]\n',
    '\n',
    '# Curated "best known examples" per type — used as fallback suggestions\n',
    '_TYPE_EXAMPLES = {\n',
    '    "mnc_product":    ["Google","Amazon","Microsoft","Meta","Adobe"],\n',
    '    "indian_product": ["Flipkart","Razorpay","Zerodha","Meesho","Groww"],\n',
    '    "it_services":    ["Infosys","TCS","Wipro","Cognizant","HCL"],\n',
    '    "mnc_finance":    ["Goldman Sachs","JPMorgan","Morgan Stanley","Citi","HDFC"],\n',
    '    "consulting":     ["Deloitte","McKinsey","BCG","Accenture","EY"],\n',
    '    "unknown":        ["Google","Flipkart","Infosys","Deloitte","Goldman Sachs"],\n',
    '}\n',
    '\n',
    '# ─────────────────────────────────────────────────────────────────────────\n',
    '# COMPANY ENRICHMENT — Approach 3 (web) + Approach 5 (Claude API)\n',
    '# These only fire when class_co() returns "unknown"\n',
    '# Existing logic is completely untouched\n',
    '# ─────────────────────────────────────────────────────────────────────────\n',
    '# ── APPROACH 3: Web search enrichment ─────────────────────────────────────────\n',
    '# Uses DuckDuckGo Instant Answer API — free, no API key needed\n',
    '# Only fires when class_co() returns "unknown"\n',
    '# Result is cached per company name to avoid repeated calls\n',
    '\n',
    'import urllib.request, urllib.parse, json as _json\n',
    '\n',
    '@st.cache_data(show_spinner=False, ttl=86400)\n',
    'def _web_search_classify(company_name: str) -> dict:\n',
    '    """\n',
    '    Search DuckDuckGo for company info and extract company type.\n',
    '    Returns {"type": str, "description": str, "source": "web"} or None.\n',
    '    """\n',
    '    try:\n',
    '        query = urllib.parse.quote(f"{company_name} company India technology")\n',
    '        url   = f"https://api.duckduckgo.com/?q={query}&format=json&no_html=1&skip_disambig=1"\n',
    '        req   = urllib.request.Request(url, headers={"User-Agent": "SalaryLensIndia/1.0"})\n',
    '        with urllib.request.urlopen(req, timeout=4) as resp:\n',
    '            data = _json.loads(resp.read().decode())\n',
    '\n',
    '        # Combine abstract + heading for analysis\n',
    '        text = (\n',
    '            (data.get("AbstractText") or "") + " " +\n',
    '            (data.get("Heading")       or "") + " " +\n',
    '            (data.get("Abstract")      or "")\n',
    '        ).lower().strip()\n',
    '\n',
    '        if not text or len(text) < 20:\n',
    '            return None\n',
    '\n',
    '        # Rule-based extraction from web text\n',
    '        co_type = None\n',
    '        tier1   = 0\n',
    '\n',
    '        # Check for FAANG/Tier-1 signals\n',
    '        if any(t in text for t in ["nasdaq","nyse","fortune 500","s&p 500","silicon valley"]):\n',
    '            co_type = "mnc_product"\n',
    '            tier1   = 1\n',
    '\n',
    '        if co_type is None:\n',
    '            # Finance signals\n',
    '            if any(w in text for w in [\n',
    '                "bank","investment bank","hedge fund","asset management",\n',
    '                "financial services","insurance","securities","stock broker",\n',
    '                "nbfc","non-banking financial"\n',
    '            ]):\n',
    '                co_type = "mnc_finance"\n',
    '\n',
    '        if co_type is None:\n',
    '            # IT Services signals\n',
    '            if any(w in text for w in [\n',
    '                "it services","outsourcing","bpo","kpo","it consulting",\n',
    '                "managed services","offshore","staffing"\n',
    '            ]):\n',
    '                co_type = "it_services"\n',
    '\n',
    '        if co_type is None:\n',
    '            # Consulting signals\n',
    '            if any(w in text for w in [\n',
    '                "management consulting","strategy consulting","advisory",\n',
    '                "professional services","big four","big 4"\n',
    '            ]):\n',
    '                co_type = "consulting"\n',
    '\n',
    '        if co_type is None:\n',
    '            # Indian product startup signals\n',
    '            if any(w in text for w in [\n',
    '                "startup","unicorn","series a","series b","series c",\n',
    '                "venture capital","backed by","founded in","indian startup",\n',
    '                "fintech","edtech","healthtech","agritech","proptech",\n',
    '                "saas","software as a service"\n',
    '            ]):\n',
    '                co_type = "indian_product"\n',
    '\n',
    '        if co_type is None:\n',
    '            # MNC product signals\n',
    '            if any(w in text for w in [\n',
    '                "software company","technology company","cloud computing",\n',
    '                "enterprise software","platform","developer tools",\n',
    '                "open source","api","developer"\n',
    '            ]):\n',
    '                co_type = "indian_product"  # default to indian_product for unknown tech\n',
    '\n',
    '        if co_type is None:\n',
    '            return None\n',
    '\n',
    '        # Get a clean description (first 120 chars)\n',
    '        desc = (data.get("AbstractText") or data.get("Abstract") or "")[:120]\n',
    '        if desc:\n',
    '            desc = desc.strip().rstrip(".") + "."\n',
    '\n',
    '        return {\n',
    '            "type":        co_type,\n',
    '            "description": desc,\n',
    '            "source":      "web",\n',
    '            "tier1":       tier1,\n',
    '        }\n',
    '\n',
    '    except Exception:\n',
    '        return None\n',
    '\n',
    '\n',
    '# ── APPROACH 5: Claude API classification ─────────────────────────────────────\n',
    '# Uses claude-haiku-3-5 — fast (~0.5s), cheap, accurate for classification\n',
    '# Only fires when web search returns None or unknown\n',
    '# Cached per company name\n',
    '\n',
    '@st.cache_data(show_spinner=False, ttl=86400)\n',
    'def _claude_classify(company_name: str) -> dict:\n',
    '    """\n',
    '    Ask Claude to classify company type using a minimal prompt.\n',
    '    Returns {"type": str, "description": str, "source": "claude"} or None.\n',
    '    """\n',
    '    try:\n',
    '        import urllib.request, json as _json\n',
    '\n',
    '        VALID_TYPES = {\n',
    '            "mnc_product", "indian_product", "it_services",\n',
    '            "mnc_finance", "consulting", "unknown"\n',
    '        }\n',
    '\n',
    '        prompt = (\n',
    '            f"Classify this company into exactly one category.\\n"\n',
    '            f"Company: {company_name}\\n\\n"\n',
    '            f"Categories:\\n"\n',
    '            f"- mnc_product: global tech product company (Google, Stripe, Datadog etc)\\n"\n',
    '            f"- indian_product: Indian startup or product company (Zerodha, Nykaa etc)\\n"\n',
    '            f"- it_services: IT outsourcing/services (TCS, Infosys etc)\\n"\n',
    '            f"- mnc_finance: bank or financial institution\\n"\n',
    '            f"- consulting: management/strategy consulting firm\\n"\n',
    '            f"- unknown: cannot determine\\n\\n"\n',
    '            f"Reply with a JSON object only, no explanation:\\n"\n',
    '            f\'{{\\"type\\": \\"<category>\\", \\"description\\": \\"<one sentence about company>\\"}}\'\n',
    '        )\n',
    '\n',
    '        payload = _json.dumps({\n',
    '            "model": "claude-haiku-3-5-20241022",\n',
    '            "max_tokens": 120,\n',
    '            "messages": [{"role": "user", "content": prompt}]\n',
    '        }).encode()\n',
    '\n',
    '        req = urllib.request.Request(\n',
    '            "https://api.anthropic.com/v1/messages",\n',
    '            data    = payload,\n',
    '            headers = {\n',
    '                "Content-Type":      "application/json",\n',
    '                "anthropic-version": "2023-06-01",\n',
    '            },\n',
    '            method = "POST"\n',
    '        )\n',
    '\n',
    '        with urllib.request.urlopen(req, timeout=6) as resp:\n',
    '            data = _json.loads(resp.read().decode())\n',
    '\n',
    '        raw = data["content"][0]["text"].strip()\n',
    '        # Clean markdown fences if present\n',
    '        raw = raw.replace("```json","").replace("```","").strip()\n',
    '        parsed = _json.loads(raw)\n',
    '\n',
    '        co_type = parsed.get("type","unknown").lower().strip()\n',
    '        desc    = parsed.get("description","")[:120]\n',
    '\n',
    '        if co_type not in VALID_TYPES:\n',
    '            co_type = "unknown"\n',
    '\n',
    '        return {\n',
    '            "type":        co_type,\n',
    '            "description": desc,\n',
    '            "source":      "claude",\n',
    '            "tier1":       1 if co_type == "mnc_product" and any(\n',
    '                t in company_name.lower() for t in [\n',
    '                    "stripe","airbnb","datadog","cloudflare","snowflake","databricks"\n',
    '                ]) else 0,\n',
    '        }\n',
    '\n',
    '    except Exception:\n',
    '        return None\n',
    '\n',
    '\n',
    '# ── SMART CLASSIFY — orchestrator ─────────────────────────────────────────────\n',
    '# Runs all lookup layers in order, stops at first confident result.\n',
    '# Existing class_co() and enc_co() are called first (unchanged).\n',
    '# Web search and Claude are only fallbacks for "unknown" companies.\n',
    '\n',
    '@st.cache_data(show_spinner=False, ttl=3600)\n',
    'def smart_classify(company_name: str) -> dict:\n',
    '    """\n',
    '    Full enrichment pipeline for unknown companies.\n',
    '    Returns dict with keys: type, enc, tier1, description, source, enriched\n',
    '    """\n',
    '    if not isinstance(company_name, str) or not company_name.strip():\n',
    '        return {"type":"unknown","enc":"other","tier1":0,\n',
    '                "description":"","source":"none","enriched":False}\n',
    '\n',
    '    # Layer 1: existing lookup (87 named + 70 extended + keyword)\n',
    '    co_type = class_co(company_name)\n',
    '    co_enc  = enc_co(company_name)\n',
    '    co_t1   = is_tier1(company_name)\n',
    '\n',
    '    if co_type != "unknown":\n',
    '        return {"type":co_type,"enc":co_enc,"tier1":co_t1,\n',
    '                "description":"","source":"lookup","enriched":False}\n',
    '\n',
    '    # Layer 2: web search (DuckDuckGo, free, no key)\n',
    '    web_result = _web_search_classify(company_name)\n',
    '    if web_result and web_result["type"] != "unknown":\n',
    '        return {\n',
    '            "type":        web_result["type"],\n',
    '            "enc":         "other",\n',
    '            "tier1":       web_result.get("tier1", 0),\n',
    '            "description": web_result["description"],\n',
    '            "source":      "web",\n',
    '            "enriched":    True,\n',
    '        }\n',
    '\n',
    '    # Layer 3: Claude API (only if web search found nothing)\n',
    '    claude_result = _claude_classify(company_name)\n',
    '    if claude_result and claude_result["type"] != "unknown":\n',
    '        return {\n',
    '            "type":        claude_result["type"],\n',
    '            "enc":         "other",\n',
    '            "tier1":       claude_result.get("tier1", 0),\n',
    '            "description": claude_result["description"],\n',
    '            "source":      "claude",\n',
    '            "enriched":    True,\n',
    '        }\n',
    '\n',
    '    # All layers exhausted — return unknown with description if any\n',
    '    desc = ""\n',
    '    if web_result:    desc = web_result.get("description","")\n',
    '    if claude_result: desc = claude_result.get("description","") or desc\n',
    '\n',
    '    return {"type":"unknown","enc":"other","tier1":0,\n',
    '            "description":desc,"source":"none","enriched":False}\n',
    '\n',
    '# ─────────────────────────────────────────────────────────────────────────\n',
    'def run_model(job_title,company,yoe_val,city,sen_override=None):\n',
    '    role=norm_role(job_title)\n',
    '    si=int(np.clip(sen_override if sen_override is not None else ext_sen(job_title),0,7))\n',
    '    ct=ext_city(city+", India")\n',
    '    yoe_f=float(yoe_val); logy=np.log1p(yoe_f)\n',
    '    eb=int(pd.cut([yoe_f],bins=[-1,2,5,10,15,40],labels=[0,1,2,3,4])[0] or 2)\n',
    '    skl=ext_skills(job_title)\n',
    '    # Use smart_classify — runs web+Claude enrichment only for unknown companies\n',
    '    # For known companies, returns immediately from local lookup (zero latency)\n',
    '    _cls = smart_classify(company)\n',
    '    _co_type    = _cls["type"]\n',
    '    _co_enc     = _cls["enc"]\n',
    '    _co_tier1   = _cls["tier1"]\n',
    '    _enriched   = _cls["enriched"]\n',
    '    _enrich_src = _cls["source"]\n',
    '    _enrich_desc= _cls["description"]\n',
    '\n',
    '    row={\n',
    '        "normalized_role":role,"company_type":_co_type,\n',
    '        "company_enc":_co_enc,"city_tier":CITY_TIER.get(ct,2),\n',
    '        "edu_level":2,"seniority_int":si,"yoe":yoe_f,\n',
    '        "has_yac":0,"is_tech_hub":int(ct in TECH_HUBS),"is_remote":0,\n',
    '        "is_tier1_mnc":_co_tier1,\n',
    '        "log_yoe":logy,"exp_bkt":eb,"sen_x_exp":si*logy,\n',
    '        "yr_feat":YEAR_W.get(2025,1.2),**skl,\n',
    '    }\n',
    '    rl=pd.DataFrame([row])\n',
    '    for col in CAT_COLS:\n',
    '        le=label_enc[col]; val=str(rl.get(col,pd.Series(["unknown"])).iloc[0])\n',
    '        val=val if val in set(le.classes_) else le.classes_[0]\n',
    '        rl[col]=le.transform([val])[0]\n',
    '    for col in FEATS:\n',
    '        if col not in rl.columns: rl[col]=0.0\n',
    '    emb_txt = role + " " + job_title + " " + SEN_LABELS.get(si, "Mid")\n',
    '    ec=pca.transform(get_emb().encode([emb_txt]).reshape(1,-1))\n',
    '    for i,col in enumerate(emb_cols): rl[col]=ec[0,i]\n',
    '    l25=lgbm_m[0.25].predict(rl[FEATS])[0]\n',
    '    l50=lgbm_m[0.50].predict(rl[FEATS])[0]\n',
    '    l75=lgbm_m[0.75].predict(rl[FEATS])[0]\n',
    '    if cb_m:\n',
    '        rc=pd.DataFrame([row])\n',
    '        for col in FEATS:\n',
    '            if col not in rc.columns: rc[col]=0.0\n',
    '        for i,col in enumerate(emb_cols): rc[col]=ec[0,i]\n',
    '        l25=0.6*l25+0.4*cb_m[0.25].predict(rc)[0]\n',
    '        l50=0.6*l50+0.4*cb_m[0.50].predict(rc)[0]\n',
    '        l75=0.6*l75+0.4*cb_m[0.75].predict(rc)[0]\n',
    '    a,b,c=mono([l25],[l50],[l75])\n',
    '    _conf = (\n',
    '        "high"   if _co_type != "unknown" and role != "Other" else\n',
    '        "medium" if role != "Other" else\n',
    '        "low"\n',
    '    )\n',
    '    return {\n',
    '        "p25":round(np.expm1(a[0])/1e5,1),"p50":round(np.expm1(b[0])/1e5,1),\n',
    '        "p75":round(np.expm1(c[0])/1e5,1),"role":role,\n',
    '        "seniority":SEN_LABELS.get(si,"Mid"),"si":si,\n',
    '        "company_type":_co_type,"city":ct,\n',
    '        "tier":CITY_TIER.get(ct,2),"is_hub":int(ct in TECH_HUBS),\n',
    '        "conf":_conf,\n',
    '        # Enrichment metadata — used by UI to show source badge\n',
    '        "enriched":    _enriched,\n',
    '        "enrich_src":  _enrich_src,\n',
    '        "enrich_desc": _enrich_desc,\n',
    '    }\n',
    '\n',
    'MATRIX_ROLES = [\n',
    '    "Data Scientist","ML Engineer","Software Engineer","Data Engineer",\n',
    '    "Data Analyst","Product Manager","DevOps/SRE",\n',
    '    "Backend Engineer","Frontend Engineer","Full Stack Engineer",\n',
    '    "Engineering Manager","Solution Architect",\n',
    ']\n',
    '# Matrix display subset — keep table readable (7 roles max)\n',
    'MATRIX_DISPLAY_ROLES = [\n',
    '    "Data Scientist","ML Engineer","Software Engineer",\n',
    '    "Data Engineer","Data Analyst","Product Manager","DevOps/SRE",\n',
    ']\n',
    'MATRIX_COS   = [\n',
    '    ("Google",          "linear-gradient(135deg,#6366f1,#8b5cf6)"),\n',
    '    ("Amazon",          "linear-gradient(135deg,#f59e0b,#fbbf24)"),\n',
    '    ("Flipkart",        "linear-gradient(135deg,#0ea5e9,#38bdf8)"),\n',
    '    ("Infosys",         "linear-gradient(135deg,#22c55e,#4ade80)"),\n',
    '    ("Deloitte",        "linear-gradient(135deg,#8b5cf6,#a78bfa)"),\n',
    '    ("Unknown Startup", "linear-gradient(135deg,#ec4899,#f472b6)"),\n',
    ']\n',
    '\n',
    'COMPANY_CHOICES = {\n',
    '    "FAANG / Tier-1 MNC": ["Google","Amazon","Microsoft","Meta","Apple","Netflix","Nvidia","Adobe","Salesforce","LinkedIn"],\n',
    '    "Indian Product":      ["Flipkart","Meesho","Razorpay","PhonePe","Swiggy","Zomato","CRED","Groww","Paytm","Freshworks"],\n',
    '    "IT Services":         ["Infosys","TCS","Wipro","HCL","Tech Mahindra","Cognizant","Accenture","LTIMindtree","Mphasis"],\n',
    '    "MNC Finance":         ["Goldman Sachs","JPMorgan","Morgan Stanley","Citi","Barclays","Deutsche Bank"],\n',
    '    "Consulting":          ["Deloitte","McKinsey","BCG","Bain","PwC","KPMG","EY","ThoughtWorks"],\n',
    '}\n',
    '\n',
    'CO_TYPE_COLORS = {\n',
    '    "mnc_product":    "#6366f1",\n',
    '    "indian_product": "#0ea5e9",\n',
    '    "it_services":    "#22c55e",\n',
    '    "mnc_finance":    "#f59e0b",\n',
    '    "consulting":     "#8b5cf6",\n',
    '    "unknown":        "#94a3b8",\n',
    '}\n',
    '\n',
    'def card(lbl,val,bg,caption="",delta=""):\n',
    '    h = (f\'<div style="background:{bg};border-radius:16px;padding:1.5rem;text-align:center;\'\n',
    '         f\'box-shadow:0 8px 24px rgba(0,0,0,0.12);margin-bottom:8px">\'\n',
    '         f\'<div style="font-size:11px;font-weight:600;letter-spacing:0.1em;\'\n',
    '         f\'color:rgba(255,255,255,0.75);text-transform:uppercase;margin-bottom:8px">{lbl}</div>\'\n',
    '         f\'<div style="font-size:32px;font-weight:800;color:white;line-height:1">{val}</div>\')\n',
    '    if caption: h += f\'<div style="font-size:12px;color:rgba(255,255,255,0.65);margin-top:6px">{caption}</div>\'\n',
    '    if delta:   h += f\'<div style="font-size:12px;color:rgba(255,255,255,0.85);margin-top:4px;font-weight:600">{delta}</div>\'\n',
    "    h += '</div>'\n",
    '    return h\n',
    '\n',
    'def mini_card(lbl,val,bg):\n',
    '    return (f\'<div style="background:{bg};border-radius:12px;padding:1rem;text-align:center;margin-bottom:6px">\'\n',
    '            f\'<div style="font-size:10px;color:rgba(255,255,255,0.7);text-transform:uppercase;letter-spacing:0.07em">{lbl}</div>\'\n',
    '            f\'<div style="font-size:22px;font-weight:800;color:white;margin-top:4px">{val}</div></div>\')\n',
    '\n',
    'def pill(txt,bg,fg):\n',
    '    return (f\'<span style="background:{bg};color:{fg};padding:5px 14px;border-radius:999px;\'\n',
    '            f\'font-size:13px;font-weight:600;margin:3px;display:inline-block">{txt}</span>\')\n',
    '\n',
    'def sig_card(lbl,val,bg):\n',
    '    return (f\'<div style="background:{bg};border-radius:10px;padding:12px 8px;text-align:center">\'\n',
    '            f\'<div style="font-size:9px;color:rgba(255,255,255,0.7);text-transform:uppercase;letter-spacing:0.08em">{lbl}</div>\'\n',
    '            f\'<div style="font-size:13px;font-weight:700;color:white;margin-top:3px">{val}</div></div>\')\n',
    '\n',
    'def row_item(lbl,val,vc):\n',
    '    return (f\'<div style="display:flex;justify-content:space-between;padding:9px 14px;\'\n',
    '            f\'background:#f8fafc;border-radius:8px;margin-bottom:6px;border:1px solid #e2e8f0">\'\n',
    '            f\'<span style="font-weight:500;color:#1e293b">{lbl}</span>\'\n',
    '            f\'<span style="color:{vc};font-weight:700">{val}</span></div>\')\n',
    '\n',
    'def tip_box(text):\n',
    '    return (f\'<div style="background:#f0fdf4;border-left:4px solid #22c55e;border-radius:8px;\'\n',
    '            f\'padding:12px 16px;margin:8px 0;font-size:14px;color:#15803d;font-weight:500">💡 {text}</div>\')\n',
    '\n',
    'def alert_box(text,bg,bd,fg):\n',
    '    return (f\'<div style="background:{bg};border-left:4px solid {bd};border-radius:8px;\'\n',
    '            f\'padding:12px 16px;margin:12px 0;font-size:14px;color:{fg};font-weight:500">{text}</div>\')\n',
    '\n',
    'def range_bar(co_name,p25,p50,p75,max_val,co_color):\n',
    '    bar_w  = int((p75-p25)/max_val*100)\n',
    '    bar_s  = int(p25/max_val*100)\n',
    '    mid_p  = int((p50-p25)/(p75-p25+0.01)*bar_w)\n',
    '    return (f\'<div style="margin-bottom:14px">\'\n',
    '            f\'<div style="display:flex;justify-content:space-between;margin-bottom:4px">\'\n',
    '            f\'<span style="font-size:13px;font-weight:600;color:#1e293b">{co_name}</span>\'\n',
    '            f\'<span style="font-size:13px;color:{co_color};font-weight:700">₹{p50}L \'\n',
    '            f\'<span style="color:#94a3b8;font-weight:400;font-size:11px">(₹{p25}–{p75}L)</span></span></div>\'\n',
    '            f\'<div style="background:#f1f5f9;border-radius:999px;height:10px;position:relative">\'\n',
    '            f\'<div style="position:absolute;left:{bar_s}%;width:{bar_w}%;height:100%;\'\n',
    '            f\'background:{co_color};opacity:0.25;border-radius:999px"></div>\'\n',
    '            f\'<div style="position:absolute;left:{bar_s+mid_p//4}%;width:3px;height:100%;\'\n',
    '            f\'background:{co_color};border-radius:999px"></div></div></div>\')\n',
    '\n',
    'def heat_color(val,vmin,vmax):\n',
    '    ratio=(val-vmin)/(vmax-vmin+0.01)\n',
    '    if ratio>0.75:   return "#dcfce7","#15803d"\n',
    '    elif ratio>0.50: return "#fef9c3","#a16207"\n',
    '    elif ratio>0.25: return "#fff7ed","#c2410c"\n',
    '    else:            return "#fef2f2","#b91c1c"\n',
    '\n',
    '# ── HEADER ─────────────────────────────────────────────────────────────────────\n',
    'st.markdown("""\n',
    '<div style="padding:1.5rem 0 0.5rem">\n',
    '  <div style="font-size:36px;font-weight:800;color:#0f172a">💰 SalaryLens India</div>\n',
    '  <div style="font-size:15px;color:#64748b;margin-top:6px">\n',
    '    Real salary intelligence — built on\n',
    '    <b style="color:#6366f1">177K+ verified data points</b>\n',
    '    from Levels.fyi · Glassdoor · Job Postings\n',
    '  </div>\n',
    '</div>\n',
    '""", unsafe_allow_html=True)\n',
    '\n',
    'sc1,sc2,sc3,sc4 = st.columns(4)\n',
    'for col,lbl,val,bg in [\n',
    '    (sc1,"Data Points","177K+","linear-gradient(135deg,#6366f1,#8b5cf6)"),\n',
    '    (sc2,"Cities","11","linear-gradient(135deg,#0ea5e9,#38bdf8)"),\n',
    '    (sc3,"Companies","200+","linear-gradient(135deg,#22c55e,#4ade80)"),\n',
    '    (sc4,"Roles","18","linear-gradient(135deg,#f59e0b,#fbbf24)"),\n',
    ']:\n',
    '    col.markdown(card(lbl,val,bg),unsafe_allow_html=True)\n',
    '\n',
    'st.markdown("<hr style=\'border:none;border-top:2px solid #f1f5f9\'>",unsafe_allow_html=True)\n',
    '\n',
    'tab1,tab2,tab3,tab4 = st.tabs([\n',
    '    "🔍  Predict Salary",\n',
    '    "🏢  Company Compare",\n',
    '    "📊  Salary Band Matrix",\n',
    '    "🌆  City Compare",\n',
    '])\n',
    '\n',
    '# ── TAB 1: PREDICT ─────────────────────────────────────────────────────────────\n',
    'with tab1:\n',
    '    st.markdown("### Enter Your Details")\n',
    '    r1c1,r1c2,r1c3 = st.columns(3)\n',
    '    with r1c1: job_title   = st.text_input("🎯 Job Title",  placeholder="e.g. Senior Data Scientist")\n',
    '    with r1c2: company     = st.text_input("🏢 Company",    placeholder="e.g. Google, Flipkart")\n',
    '    with r1c3: city        = st.selectbox("📍 City",["Bengaluru","Hyderabad","Pune","Gurgaon",\n',
    '                                "Noida","Mumbai","Chennai","Delhi","Ahmedabad","Kolkata","Kochi"])\n',
    '    r2c1,r2c2,r2c3 = st.columns(3)\n',
    '    with r2c1: yoe         = st.number_input("💼 Years of Experience",0,30,3)\n',
    '    with r2c2: current_sal = st.number_input("💰 Current CTC (LPA) — optional",0.0,200.0,0.0,0.5)\n',
    '    with r2c3:\n',
    '        sen_opts = {v:k for k,v in SEN_LABELS.items()}\n',
    '        sen_sel  = st.selectbox("🎓 Seniority",["Auto-detect"]+list(SEN_LABELS.values()))\n',
    '        sen_ov   = None if sen_sel=="Auto-detect" else sen_opts[sen_sel]\n',
    '    predict_btn = st.button("🔍  Get Salary Range",use_container_width=True,type="primary",key="pred_btn")\n',
    '\n',
    '    if predict_btn:\n',
    '        if not job_title:\n',
    '            st.warning("Please enter a Job Title.")\n',
    '            st.stop()\n',
    '\n',
    '        # ── UNKNOWN COMPANY GATE ──────────────────────────────────────────────\n',
    '        # Before running the model, check if company is known.\n',
    '        # If completely unknown after all enrichment layers, stop and show\n',
    '        # a clear "no data" message with similar company suggestions.\n',
    '        # This only applies when the user actually typed a company name.\n',
    '        _co_input = (company or "").strip()\n',
    '        if _co_input:\n',
    '            with st.spinner(f"Looking up {_co_input}..."):\n',
    '                _cls_check = smart_classify(_co_input)\n',
    '\n',
    '            if _cls_check["type"] == "unknown":\n',
    '                # Company is fully unknown — hard stop, no prediction\n',
    '                st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1rem 0\'>",unsafe_allow_html=True)\n',
    '                st.markdown(\n',
    '                    f\'<div style="background:#fef2f2;border:1.5px solid #fca5a5;border-radius:14px;\' +\n',
    '                    f\'padding:1.4rem 1.6rem;margin-bottom:1rem">\' +\n',
    '                    f\'<div style="font-size:18px;font-weight:700;color:#b91c1c;margin-bottom:6px">\' +\n',
    "                    f'❌  No data available for &quot;{_co_input}&quot;</div>' +\n",
    '                    f\'<div style="font-size:14px;color:#7f1d1d;line-height:1.7">\' +\n',
    "                    f'This company is not in our dataset and could not be classified ' +\n",
    "                    f'by web search or AI. We cannot make a reliable salary prediction ' +\n",
    "                    f'without knowing the company type — predictions would be meaningless.</div>' +\n",
    "                    f'</div>',\n",
    '                    unsafe_allow_html=True)\n',
    '\n',
    '                # Similar company suggestions\n',
    '                _suggestions = _suggest_similar(_co_input, n=5)\n',
    '                if _suggestions:\n',
    '                    st.markdown("#### 💡 Try one of these similar companies instead:")\n',
    '                    sug_cols = st.columns(len(_suggestions))\n',
    '                    for col, (sug_co, sug_type) in zip(sug_cols, _suggestions):\n',
    '                        sug_label = sug_type.replace("_"," ").title()\n',
    '                        sug_color = CO_TYPE_COLORS.get(sug_type, "#94a3b8")\n',
    '                        col.markdown(\n',
    '                            f\'<div style="border:1.5px solid {sug_color};border-radius:12px;\' +\n',
    '                            f\'padding:12px;text-align:center;cursor:pointer">\' +\n',
    '                            f\'<div style="font-size:14px;font-weight:700;color:#0f172a">{sug_co}</div>\' +\n',
    '                            f\'<div style="font-size:11px;color:{sug_color};margin-top:4px;font-weight:600">{sug_label}</div>\' +\n',
    "                            f'</div>',\n",
    '                            unsafe_allow_html=True)\n',
    '\n',
    '                # Also show what we do know for the role + city (no company)\n',
    '                st.markdown("<br>",unsafe_allow_html=True)\n',
    '                st.markdown(\n',
    '                    f\'<div style="background:#f0f9ff;border:1px solid #bae6fd;border-radius:12px;\' +\n',
    '                    f\'padding:1rem 1.4rem;font-size:14px;color:#0369a1">\' +\n',
    "                    f'💡 <b>Alternatively:</b> leave the Company field empty to get the ' +\n",
    "                    f'<b>market average salary</b> for {job_title} in {city} without ' +\n",
    "                    f'company-specific adjustment.</div>',\n",
    '                    unsafe_allow_html=True)\n',
    '                st.stop()\n',
    '\n',
    '        # Company is known (or no company entered) — proceed with prediction\n',
    '        with st.spinner("Analysing 177K+ data points..."):\n',
    '            r = run_model(job_title,company or "unknown",yoe,city,sen_ov)\n',
    '        p25,p50,p75 = r["p25"],r["p50"],r["p75"]\n',
    '        span        = round(p75-p25,1)\n',
    '        co_label    = r["company_type"].replace("_"," ").title()\n',
    '        st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1rem 0\'>",unsafe_allow_html=True)\n',
    '        # Build pill row — same as before + enrichment badge if company was enriched\n',
    '        _conf_pill = pill(\n',
    '            "✅ High Confidence" if r["conf"]=="high" else "⚠️ Medium Confidence",\n',
    '            "#f0fdf4" if r["conf"]=="high" else "#fef9c3",\n',
    '            "#15803d" if r["conf"]=="high" else "#a16207"\n',
    '        )\n',
    '        _enrich_pill = ""\n',
    '        if r.get("enriched"):\n',
    '            _src = r.get("enrich_src","")\n',
    '            if _src == "web":\n',
    '                _enrich_pill = pill("🌐 Company info via web search","#e0f2fe","#0369a1")\n',
    '            elif _src == "claude":\n',
    '                _enrich_pill = pill("🤖 Company classified by AI","#ede9fe","#5b21b6")\n',
    '        ph=(pill(f"🎯 {r[\'role\']}","#ede9fe","#5b21b6")+\n',
    '            pill(f"📊 {r[\'seniority\']}","#dcfce7","#15803d")+\n',
    '            pill(f"🏢 {co_label}","#fff7ed","#c2410c")+\n',
    '            pill(f"📍 {r[\'city\'].title()} T{r[\'tier\']}","#f0f9ff","#0369a1")+\n',
    '            _conf_pill + _enrich_pill)\n',
    '        st.markdown(ph,unsafe_allow_html=True)\n',
    '        # Show enrichment description if available\n',
    '        if r.get("enrich_desc"):\n',
    '            st.markdown(\n',
    '                f\'<div style="background:#f8fafc;border-left:3px solid #cbd5e1;\'\n',
    "                f'border-radius:0 6px 6px 0;padding:8px 14px;margin:4px 0 8px;'\n",
    '                f\'font-size:12px;color:#64748b">📖 {r["enrich_desc"]}</div>\',\n',
    '                unsafe_allow_html=True)\n',
    '        st.markdown("")\n',
    '        c1,c2,c3=st.columns(3)\n',
    '        c1.markdown(card("P25 — Floor",   f"₹{p25} LPA","linear-gradient(135deg,#3b82f6,#60a5fa)","25% earn below this"),unsafe_allow_html=True)\n',
    '        c2.markdown(card("P50 — Expected",f"₹{p50} LPA","linear-gradient(135deg,#22c55e,#4ade80)","Market median",f"+₹{round(p50-p25,1)}L above P25"),unsafe_allow_html=True)\n',
    '        c3.markdown(card("P75 — Ceiling", f"₹{p75} LPA","linear-gradient(135deg,#f59e0b,#fbbf24)","Top 25% of earners",f"+₹{round(p75-p50,1)}L above P50"),unsafe_allow_html=True)\n',
    '        st.progress((p50-p25)/(span+0.01),text=f"P50 ₹{p50}L at {round((p50-p25)/(span+0.01)*100)}% of range  |  ₹{p25}L to ₹{p75}L  |  Span ₹{span}L")\n',
    '        st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1.5rem 0\'>",unsafe_allow_html=True)\n',
    '\n',
    '        if current_sal > 0:\n',
    '            gap=round(p50-current_sal,1)\n',
    '            st.markdown("### 📊 Are You Underpaid?")\n',
    '            uc1,uc2,uc3=st.columns(3)\n',
    '            uc1.markdown(card("Your Current CTC",f"₹{current_sal}L","linear-gradient(135deg,#64748b,#94a3b8)"),unsafe_allow_html=True)\n',
    '            uc2.markdown(card("Market Median",   f"₹{p50}L",       "linear-gradient(135deg,#6366f1,#8b5cf6)"),unsafe_allow_html=True)\n',
    '            uc3.markdown(card("Gap vs Market",\n',
    '                f"{\'−\' if gap>0 else \'+\'}₹{abs(gap)}L",\n',
    '                "linear-gradient(135deg,#ef4444,#f87171)" if gap>2 else\n',
    '                "linear-gradient(135deg,#f59e0b,#fbbf24)" if gap>0 else\n',
    '                "linear-gradient(135deg,#22c55e,#4ade80)",\n',
    '                "Underpaid" if gap>2 else "Slightly below" if gap>0 else "Well compensated"),\n',
    '                unsafe_allow_html=True)\n',
    '            if gap>2:   st.markdown(alert_box(f"⚠️ ₹{gap}L below market median. Strong case for a raise or switch.","#fef2f2","#ef4444","#b91c1c"),unsafe_allow_html=True)\n',
    '            elif gap>0: st.markdown(alert_box(f"⚡ Slightly below by ₹{gap}L. Raise at next review.","#fef9c3","#eab308","#92400e"),unsafe_allow_html=True)\n',
    '            else:       st.markdown(alert_box(f"✅ ₹{abs(gap)}L above market median. Well positioned.","#f0fdf4","#22c55e","#15803d"),unsafe_allow_html=True)\n',
    '            st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1.5rem 0\'>",unsafe_allow_html=True)\n',
    '\n',
    '        st.markdown(f"### 📈 Salary Growth Curve — {r[\'role\']}")\n',
    '        st.caption(f"{r[\'city\'].title()} · {co_label}")\n',
    '        pts=[run_model(job_title,company or "unknown",yr,city)["p50"] for yr in range(0,16)]\n',
    '        st.line_chart(pd.DataFrame({"Experience (Years)":range(0,16),"Expected Salary (LPA)":pts}).set_index("Experience (Years)"))\n',
    '        st.caption(f"At {yoe} yrs: ₹{p50}L  ·  At 5: ₹{pts[5]}L  ·  At 10: ₹{pts[10]}L  ·  At 15: ₹{pts[15]}L")\n',
    '        st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1.5rem 0\'>",unsafe_allow_html=True)\n',
    '\n',
    '        st.markdown("### 🔑 Key Signals")\n',
    '        sg=st.columns(6)\n',
    '        for col,lbl,val,bg in [\n',
    '            (sg[0],"Role",      r["role"],                      "linear-gradient(135deg,#6366f1,#8b5cf6)"),\n',
    '            (sg[1],"Seniority", r["seniority"],                 "linear-gradient(135deg,#0ea5e9,#38bdf8)"),\n',
    '            (sg[2],"Company",   co_label,                       "linear-gradient(135deg,#f59e0b,#fbbf24)"),\n',
    '            (sg[3],"City Tier", f"Tier {r[\'tier\']}",            "linear-gradient(135deg,#22c55e,#4ade80)"),\n',
    '            (sg[4],"Experience",f"{yoe} yrs",                   "linear-gradient(135deg,#8b5cf6,#a78bfa)"),\n',
    '            (sg[5],"Tech Hub",  "Yes ✅" if r["is_hub"] else "No","linear-gradient(135deg,#ec4899,#f472b6)"),\n',
    '        ]:\n',
    '            col.markdown(sig_card(lbl,val,bg),unsafe_allow_html=True)\n',
    '        st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1.5rem 0\'>",unsafe_allow_html=True)\n',
    '\n',
    '        if shap_imp:\n',
    '            st.markdown("### 🧠 What Drove This Prediction")\n',
    '            top6=sorted(shap_imp.items(),key=lambda x:-x[1])[:6]\n',
    '            max_v=top6[0][1]\n',
    '            for feat,val in top6:\n',
    '                clean=feat.replace("sk_","Skill: ").replace("_"," ").title()\n',
    '                pct=int(val/max_v*100)\n',
    '                st.markdown(\n',
    '                    f\'<div style="margin-bottom:12px">\'\n',
    '                    f\'<div style="display:flex;justify-content:space-between;margin-bottom:5px">\'\n',
    '                    f\'<span style="font-size:13px;font-weight:600;color:#374151">{clean}</span>\'\n',
    '                    f\'<span style="font-size:13px;color:#6366f1;font-weight:700">{pct}%</span></div>\'\n',
    '                    f\'<div style="background:#e5e7eb;border-radius:999px;height:8px">\'\n',
    '                    f\'<div style="background:linear-gradient(90deg,#6366f1,#8b5cf6);width:{pct}%;height:100%;border-radius:999px"></div></div></div>\',\n',
    '                    unsafe_allow_html=True)\n',
    '            st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1.5rem 0\'>",unsafe_allow_html=True)\n',
    '\n',
    '        st.markdown("### 💼 Negotiation Intelligence")\n',
    '        co2=r["company_type"]; si2=r["si"]\n',
    '        for t in [\n',
    '            {"it_services":"IT services firms have rigid bands — use Levels.fyi data to justify above-band offers.",\n',
    '             "mnc_product":"MNC product companies have signing bonus flexibility — negotiate that separately from base.",\n',
    '             "indian_product":"Indian product startups offer ESOPs — push for accelerated vesting if base is capped.",\n',
    '             "consulting":"Consulting firms move in cohorts — focus on joining bonus and early grade acceleration."\n',
    '            }.get(co2,"Anchor on market P75 and let them negotiate down. Never reveal your current salary first."),\n',
    '            f"{r[\'city\'].title()} is a top tech hub — demand the hub premium on top of base." if r["is_hub"]\n',
    '            else "Non-metro: negotiate remote work option or explicit relocation allowance.",\n',
    '            "At Staff/Lead+ level: total comp (ESOPs + bonus) matters more than base." if si2>=4\n',
    '            else "Early career: prioritise fast promotion clause, learning budget, and 6-month review."\n',
    '        ]:\n',
    '            st.markdown(tip_box(t),unsafe_allow_html=True)\n',
    '\n',
    '    else:\n',
    '        st.markdown("""\n',
    '> **How to use:** Enter your job title, company, city and years of experience on the left,\n',
    '> then click **Get Salary Range**. The model returns a P25 / P50 / P75 salary band —\n',
    '> the floor, expected median, and top-of-range for your profile.\n',
    '> Optionally enter your current CTC to see if you are underpaid vs the market.\n',
    '""")\n',
    '        st.markdown("## 🌏 Indian Tech Salary Landscape")\n',
    '        mi1,mi2,mi3=st.columns(3)\n',
    '        for col,title,rows in [\n',
    '            (mi1,"🏆 Highest Paying Roles",\n',
    '             [("ML Engineer","₹42L","#6366f1"),("Product Manager","₹40L","#6366f1"),\n',
    '              ("Data Scientist","₹38L","#6366f1"),("DevOps/SRE","₹35L","#6366f1"),\n',
    '              ("Software Engineer","₹32L","#6366f1")]),\n',
    '            (mi2,"📍 Highest Paying Cities",\n',
    '             [("Bengaluru","₹38L","#0ea5e9"),("Mumbai","₹36L","#0ea5e9"),\n',
    '              ("Hyderabad","₹34L","#0ea5e9"),("Gurgaon","₹33L","#0ea5e9"),\n',
    '              ("Pune","₹30L","#0ea5e9")]),\n',
    '            (mi3,"🏢 Company Type Premium",\n',
    '             [("MNC Product","+85%","#22c55e"),("MNC Finance","+78%","#22c55e"),\n',
    '              ("Indian Product","+45%","#22c55e"),("Consulting","+30%","#22c55e"),\n',
    '              ("IT Services","Baseline","#64748b")]),\n',
    '        ]:\n',
    '            with col:\n',
    '                st.markdown(f"#### {title}")\n',
    '                for lbl,val,vc in rows:\n',
    '                    st.markdown(row_item(lbl,val,vc),unsafe_allow_html=True)\n',
    '\n',
    '# ── TAB 2: COMPANY COMPARE ──────────────────────────────────────────────────────\n',
    'with tab2:\n',
    '    st.markdown("### Compare Salary Ranges Across Companies")\n',
    '    st.markdown("""\n',
    '> **How it works:** Select a role, city and experience level. Pick from 40+ known companies\n',
    '> using the checkboxes below — or type any company name in the **Custom** box.\n',
    '> The model predicts P25 / P50 / P75 salary ranges for each company based on its type,\n',
    '> location tier and FAANG classification. Results are ranked by median salary.\n',
    '""")\n',
    '\n',
    '    cc1,cc2,cc3=st.columns(3)\n',
    '    with cc1: cmp_role=st.selectbox("🎯 Role",MATRIX_ROLES,key="cmp_role")\n',
    '    with cc2: cmp_city=st.selectbox("📍 City",["Bengaluru","Hyderabad","Pune","Gurgaon","Noida","Mumbai","Chennai","Delhi","Ahmedabad","Kolkata","Kochi","Jaipur","Chandigarh","Coimbatore","Indore","Nagpur","Lucknow"],key="cmp_city")\n',
    '    with cc3: cmp_yoe =st.number_input("💼 Years of Experience",0,30,5,key="cmp_yoe")\n',
    '\n',
    '    # ── Quick-select known companies by group ─────────────────────────────────\n',
    '    st.markdown("**Quick-select known companies:**")\n',
    '    selected_cos = []\n',
    '    for grp, cos in COMPANY_CHOICES.items():\n',
    '        with st.expander(grp, expanded=(grp=="FAANG / Tier-1 MNC")):\n',
    '            grp_cols = st.columns(5)\n',
    '            for i, co in enumerate(cos):\n',
    '                with grp_cols[i % 5]:\n',
    '                    if st.checkbox(co, key=f"chk_{co}",\n',
    '                                   value=co in ["Google","Amazon","Flipkart","Infosys","Deloitte"]):\n',
    '                        selected_cos.append(co)\n',
    '\n',
    '    # ── Custom company input ──────────────────────────────────────────────────\n',
    '    st.markdown("**Or add any other company:**")\n',
    '    custom_input = st.text_input(\n',
    '        "Type company names separated by commas",\n',
    '        placeholder="e.g. Zeta, Juspay, Nykaa, Postman",\n',
    '        key="custom_cos"\n',
    '    )\n',
    '    if custom_input.strip():\n',
    '        custom_list = [c.strip() for c in custom_input.split(",") if c.strip()]\n',
    '        if custom_list:\n',
    '            st.caption(f"Will also compare: {\', \'.join(custom_list)}")\n',
    '            selected_cos = selected_cos + custom_list\n',
    '\n',
    '    if selected_cos:\n',
    '        st.caption(f"**{len(selected_cos)} companies selected** — results ranked by P50 median")\n',
    '    cmp_btn=st.button("Compare Companies →",type="primary",use_container_width=True,key="cmp_btn")\n',
    '\n',
    '    if cmp_btn and selected_cos:\n',
    '        if len(selected_cos)>8:\n',
    '            st.warning("Select 8 or fewer companies for a clean comparison.")\n',
    '        else:\n',
    '            with st.spinner(f"Computing salary ranges for {len(selected_cos)} companies..."):\n',
    '                results=[]\n',
    '                for co in selected_cos:\n',
    '                    r=run_model(cmp_role,co,cmp_yoe,cmp_city)\n',
    '                    results.append({"company":co,"co_type":r["company_type"],\n',
    '                                    "p25":r["p25"],"p50":r["p50"],"p75":r["p75"],\n',
    '                                    "color":CO_TYPE_COLORS.get(r["company_type"],"#94a3b8")})\n',
    '                results.sort(key=lambda x:-x["p50"])\n',
    '                best=results[0]; max_val=results[0]["p75"]*1.1\n',
    '\n',
    '            st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1rem 0\'>",unsafe_allow_html=True)\n',
    '            st.markdown(\n',
    '                f\'<div style="background:linear-gradient(135deg,#6366f1,#8b5cf6);border-radius:14px;padding:1.2rem 1.8rem;margin-bottom:1.5rem;display:flex;justify-content:space-between;align-items:center">\'\n',
    '                f\'<div><div style="font-size:12px;color:rgba(255,255,255,0.7);font-weight:600;text-transform:uppercase;letter-spacing:0.08em">🏆 Highest Paying — {cmp_role} · {cmp_yoe} yrs · {cmp_city}</div>\'\n',
    '                f\'<div style="font-size:24px;font-weight:800;color:white;margin-top:4px">{best["company"]}</div></div>\'\n',
    '                f\'<div style="text-align:right"><div style="font-size:32px;font-weight:800;color:white">₹{best["p50"]}L</div>\'\n',
    '                f\'<div style="font-size:12px;color:rgba(255,255,255,0.7)">P50 · Range ₹{best["p25"]}–{best["p75"]}L</div></div></div>\',\n',
    '                unsafe_allow_html=True)\n',
    '\n',
    '            st.markdown("#### Salary Range Comparison")\n',
    '            bar_html=""\n',
    '            for res in results:\n',
    '                bar_html+=range_bar(res["company"],res["p25"],res["p50"],res["p75"],max_val,res["color"])\n',
    '            st.markdown(bar_html,unsafe_allow_html=True)\n',
    '            st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1.5rem 0\'>",unsafe_allow_html=True)\n',
    '\n',
    '            st.markdown("#### Detailed Breakdown")\n',
    '            n_cols=min(4,len(results))\n',
    '            for i in range(0,len(results),n_cols):\n',
    '                batch=results[i:i+n_cols]\n',
    '                cols=st.columns(len(batch))\n',
    '                for col,res in zip(cols,batch):\n',
    '                    rank=results.index(res)+1\n',
    '                    medal={1:"🥇",2:"🥈",3:"🥉"}.get(rank,f"#{rank}")\n',
    '                    co_label=res["co_type"].replace("_"," ").title()\n',
    '                    col.markdown(\n',
    '                        f\'<div style="border:2px solid {res["color"]};border-radius:14px;padding:1.2rem;margin-bottom:8px">\'\n',
    '                        f\'<div style="display:flex;justify-content:space-between;margin-bottom:10px">\'\n',
    '                        f\'<span style="font-size:15px;font-weight:700;color:#0f172a">{res["company"]}</span>\'\n',
    '                        f\'<span style="font-size:18px">{medal}</span></div>\'\n',
    '                        f\'<div style="font-size:11px;color:#94a3b8;margin-bottom:8px">{co_label}</div>\'\n',
    '                        f\'<div style="background:#f8fafc;border-radius:8px;padding:10px">\'\n',
    '                        f\'<div style="display:flex;justify-content:space-between;margin-bottom:4px">\'\n',
    '                        f\'<span style="font-size:12px;color:#64748b">P25 Floor</span>\'\n',
    '                        f\'<span style="font-size:13px;font-weight:600;color:#3b82f6">₹{res["p25"]}L</span></div>\'\n',
    '                        f\'<div style="display:flex;justify-content:space-between;margin-bottom:4px">\'\n',
    '                        f\'<span style="font-size:12px;color:#64748b">P50 Target</span>\'\n',
    '                        f\'<span style="font-size:15px;font-weight:700;color:{res["color"]}">₹{res["p50"]}L</span></div>\'\n',
    '                        f\'<div style="display:flex;justify-content:space-between">\'\n',
    '                        f\'<span style="font-size:12px;color:#64748b">P75 Ceiling</span>\'\n',
    '                        f\'<span style="font-size:13px;font-weight:600;color:#f59e0b">₹{res["p75"]}L</span></div></div>\'\n',
    '                        + (f\'<div style="margin-top:8px;font-size:12px;color:#22c55e;font-weight:600">+₹{round(res["p50"]-results[-1]["p50"],1)}L vs lowest</div>\' if rank==1 else "")\n',
    "                        + '</div>',\n",
    '                        unsafe_allow_html=True)\n',
    '\n',
    '            st.markdown("#### Summary Table")\n',
    '            df_tbl=pd.DataFrame([{\n',
    '                "Rank":    str(results.index(r)+1),\n',
    '                "Company": r["company"],\n',
    '                "Type":    r["co_type"].replace("_"," ").title(),\n',
    '                "P25":     f"₹{r[\'p25\']}L",\n',
    '                "P50":     f"₹{r[\'p50\']}L",\n',
    '                "P75":     f"₹{r[\'p75\']}L",\n',
    '                "Range":   f"₹{round(r[\'p75\']-r[\'p25\'],1)}L span",\n',
    '                "vs Best": f"-₹{round(results[0][\'p50\']-r[\'p50\'],1)}L" if r!=results[0] else "🏆 Best",\n',
    '            } for r in results])\n',
    '            st.dataframe(df_tbl,use_container_width=True,hide_index=True)\n',
    '    elif cmp_btn:\n',
    '        st.info("Select at least one company.")\n',
    '    else:\n',
    '        st.info("Select companies above and click Compare Companies →")\n',
    '\n',
    '# ── TAB 3: SALARY BAND MATRIX ───────────────────────────────────────────────────\n',
    'with tab3:\n',
    '    st.markdown("### Salary Band Matrix — P50 by Role × Company")\n',
    '    st.markdown("""\n',
    '> **How it works:** This matrix shows the **model-predicted P50 (median) salary** for every\n',
    '> combination of role and company. It uses the same ML model as the Predict tab — so values\n',
    '> reflect how the model learned company type (FAANG vs IT Services vs Consulting),\n',
    '> city tier, and experience level, not raw averages from data.\n',
    '>\n',
    '> **Color coding:** 🟢 Green = top quartile · 🟡 Yellow = above median ·\n',
    '> 🟠 Orange = below median · 🔴 Red = bottom quartile — all relative to values in this matrix.\n',
    '>\n',
    '> ⚠️ **Note:** High-end companies (Google, Amazon) may be slightly underestimated at senior levels\n',
    '> due to data sparsity above ₹70L in the training set. Use as a directional benchmark.\n',
    '""")\n',
    '    mx1,mx2=st.columns(2)\n',
    '    with mx1: mx_city=st.selectbox("📍 City",["Bengaluru","Hyderabad","Pune","Gurgaon","Noida","Mumbai","Chennai","Delhi","Ahmedabad","Kolkata","Kochi","Jaipur","Chandigarh","Coimbatore","Indore","Nagpur","Lucknow"],key="mx_city")\n',
    '    with mx2: mx_yoe =st.number_input("💼 Years of Experience",0,30,5,key="mx_yoe")\n',
    '    gen_btn=st.button("Generate Matrix →",type="primary",use_container_width=True,key="gen_btn")\n',
    '\n',
    '    if gen_btn:\n',
    '        matrix_cos=[co for co,_ in MATRIX_COS]\n',
    '        with st.spinner("Computing salary matrix..."):\n',
    '            rows=[]\n',
    '            for role in MATRIX_DISPLAY_ROLES:\n',
    '                row={"Role":role}\n',
    '                for co in matrix_cos:\n',
    '                    r=run_model(role,co,mx_yoe,mx_city)\n',
    '                    row[co]=r["p50"]\n',
    '                rows.append(row)\n',
    '            df_matrix=pd.DataFrame(rows).set_index("Role")\n',
    '\n',
    '        st.markdown("<hr style=\'border:none;border-top:1px solid #f1f5f9;margin:1rem 0\'>",unsafe_allow_html=True)\n',
    '        st.markdown(f"#### P50 Salary (LPA) — {mx_city} · {mx_yoe} Years Experience")\n',
    '\n',
    '        all_vals=df_matrix.values.flatten()\n',
    '        vmin,vmax=all_vals.min(),all_vals.max()\n',
    '\n',
    '        # Build HTML table — no font-family quotes issue\n',
    '        tbl  = \'<table style="width:100%;border-collapse:separate;border-spacing:4px">\'\n',
    '        tbl += \'<tr><th style="text-align:left;padding:8px 12px;font-size:12px;color:#94a3b8;font-weight:600">ROLE</th>\'\n',
    '        for co,bg in MATRIX_COS:\n',
    '            tbl += f\'<th style="padding:8px 12px;border-radius:8px;background:{bg};font-size:12px;color:white;font-weight:700;text-align:center">{co}</th>\'\n',
    "        tbl += '</tr>'\n",
    '        for role in MATRIX_DISPLAY_ROLES:\n',
    '            tbl += f\'<tr><td style="padding:10px 12px;font-size:13px;font-weight:600;color:#374151;white-space:nowrap">{role}</td>\'\n',
    '            for co,_ in MATRIX_COS:\n',
    '                val=df_matrix.loc[role,co]\n',
    '                bg,fg=heat_color(val,vmin,vmax)\n',
    '                tbl += f\'<td style="padding:10px 12px;text-align:center;border-radius:8px;background:{bg};color:{fg};font-size:14px;font-weight:700">₹{val}L</td>\'\n',
    "            tbl += '</tr>'\n",
    "        tbl += '</table>'\n",
    '        st.markdown(tbl,unsafe_allow_html=True)\n',
    '        st.markdown("")\n',
    '\n',
    '        best_co  =df_matrix.mean().idxmax()\n',
    '        best_role=df_matrix[matrix_cos[0]].idxmax()\n',
    '        gap_role =(df_matrix.max(axis=1)-df_matrix.min(axis=1)).idxmax()\n',
    '        gap_val  =round((df_matrix.max(axis=1)-df_matrix.min(axis=1)).max(),1)\n',
    '\n',
    '        i1,i2,i3=st.columns(3)\n',
    '        i1.markdown(card("Highest Paying Company",best_co,"linear-gradient(135deg,#6366f1,#8b5cf6)",\n',
    '                         f"Avg ₹{df_matrix[best_co].mean():.1f}L across roles"),unsafe_allow_html=True)\n',
    '        i2.markdown(card(f"Top Role at {matrix_cos[0]}",best_role,"linear-gradient(135deg,#22c55e,#4ade80)",\n',
    '                         f"₹{df_matrix.loc[best_role,matrix_cos[0]]}L median"),unsafe_allow_html=True)\n',
    '        i3.markdown(card("Biggest Company Gap",gap_role,"linear-gradient(135deg,#f59e0b,#fbbf24)",\n',
    '                         f"₹{gap_val}L difference across companies"),unsafe_allow_html=True)\n',
    '\n',
    '        csv=df_matrix.to_csv()\n',
    '        st.download_button("⬇️  Download Matrix as CSV",data=csv,\n',
    '                           file_name=f"salary_matrix_{mx_city}_{mx_yoe}yrs.csv",mime="text/csv")\n',
    '    else:\n',
    '        st.info("Select city + experience and click Generate Matrix →")\n',
    '\n',
    '# ── TAB 4: CITY COMPARE ─────────────────────────────────────────────────────────\n',
    'with tab4:\n',
    '    st.markdown("### Compare Same Role Across Cities")\n',
    '    ct1,ct2,ct3=st.columns(3)\n',
    '    with ct1: cty_title  =st.text_input("Role",placeholder="Data Scientist",key="ct_t")\n',
    '    with ct2: cty_company=st.text_input("Company",placeholder="Amazon",key="ct_c")\n',
    '    with ct3: cty_yoe    =st.number_input("Experience (years)",0,30,3,key="ct_y")\n',
    '    cty_cities=st.multiselect("Select cities",\n',
    '                   ["Bengaluru","Hyderabad","Pune","Gurgaon","Noida","Mumbai","Chennai","Delhi"],\n',
    '                   default=["Bengaluru","Hyderabad","Pune"])\n',
    '    cty_btn=st.button("Compare Cities →",type="primary",use_container_width=True,key="cty_btn")\n',
    '\n',
    '    if cty_btn and cty_title and len(cty_cities)>=2:\n',
    '        results=[(c,run_model(cty_title,cty_company or "unknown",cty_yoe,c)) for c in cty_cities]\n',
    '        best=max(results,key=lambda x:x[1]["p50"])[0]\n',
    '        cols=st.columns(len(results))\n',
    '        for i,(city_c,r) in enumerate(results):\n',
    '            with cols[i]:\n',
    '                crown=" 🏆 Best Pay" if city_c==best else ""\n',
    '                st.markdown(f"#### 📍 {city_c}{crown}")\n',
    '                for lbl,val,bg in [\n',
    '                    ("P25 Floor",   f"₹{r[\'p25\']}L","#3b82f6"),\n',
    '                    ("P50 Expected",f"₹{r[\'p50\']}L","#22c55e"),\n',
    '                    ("P75 Ceiling", f"₹{r[\'p75\']}L","#f59e0b"),\n',
    '                ]:\n',
    '                    st.markdown(mini_card(lbl,val,bg),unsafe_allow_html=True)\n',
    '                if i>0:\n',
    '                    diff=round(r["p50"]-results[0][1]["p50"],1)\n',
    '                    sign="+" if diff>=0 else ""\n',
    '                    color="#22c55e" if diff>=0 else "#ef4444"\n',
    '                    st.markdown(f\'<div style="text-align:center;color:{color};font-weight:600;font-size:14px;margin-top:4px">{sign}₹{diff}L vs {results[0][0]}</div>\',unsafe_allow_html=True)\n',
    '    elif cty_btn:\n',
    '        st.info("Enter a role and select at least 2 cities.")\n',
    '\n',
    'st.markdown("<hr style=\'border:none;border-top:1px solid #e2e8f0;margin:2rem 0\'>",unsafe_allow_html=True)\n',
    'st.markdown(\'<div style="text-align:center;color:#94a3b8;font-size:12px;padding-bottom:1rem">SalaryLens India &nbsp;·&nbsp; Levels.fyi · Glassdoor · Job Postings &nbsp;·&nbsp; For reference only</div>\',unsafe_allow_html=True)\n',
]
with open(APP_PATH,"w") as f:
    f.writelines(app_lines)
print(f"App written: {APP_PATH} ({len(app_lines)} lines)")

subprocess.run([sys.executable,"-m","pip","install","streamlit","pyngrok","-q",
                "--break-system-packages"],capture_output=True)
subprocess.run(["pkill","-f","streamlit"],capture_output=True)
time.sleep(2)
subprocess.Popen(
    [sys.executable,"-m","streamlit","run",APP_PATH,
     "--server.port=8501","--server.headless=true",
     "--server.enableCORS=false","--server.enableXsrfProtection=false"],
    stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(5)
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill(); time.sleep(1)
url = ngrok.connect(8501)
print(f"\n{chr(61)*52}")
print(f"  SALARYLENS INDIA IS LIVE")
print(f"  {url}")
print(f"{chr(61)*52}")
print("  No password needed.")